# 05 · Segmentation

Extracted from `Segementation .ipynb` (89 cells). **All outputs preserved.** The original notebook is unmodified.

Calcification-specific work, CDD/CSAW ingestion, combined multi-dataset runs and `pip install` cells were left in the original — they are not part of the mass thesis.


## A · Final architecture — the model in the thesis


**original cell 63** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [3]:
# ══════════════════════════════════════════════════════════════════════
# NOVELTY — Deeply-Supervised Attention U-Net + ASPP bottleneck
#   CLAHE + 8x augmentation + Tversky(0.7,0.3)
#   vs your plain attention U-Net baseline (same splits, same everything else)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8; THR=0.5; TV_A,TV_B=0.7,0.3
DS_WEIGHTS=[1.0,0.5,0.3,0.2]   # deep-supervision loss weights: main, d2, d3, d4
torch.backends.cudnn.benchmark=True

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d,aug,mult=1): s.df=d.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))

# --- ASPP bottleneck: atrous pyramid captures multi-scale lesion context ---
class ASPP(nn.Module):
    def __init__(s,i,o):
        super().__init__()
        s.b0=nn.Sequential(nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b1=nn.Sequential(nn.Conv2d(i,o,3,padding=6,dilation=6),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b2=nn.Sequential(nn.Conv2d(i,o,3,padding=12,dilation=12),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b3=nn.Sequential(nn.Conv2d(i,o,3,padding=18,dilation=18),nn.BatchNorm2d(o),nn.ReLU(True))
        s.gp=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.proj=nn.Sequential(nn.Conv2d(o*5,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(s,x):
        g=F.interpolate(s.gp(x),size=x.shape[2:],mode="bilinear",align_corners=False)
        return s.proj(torch.cat([s.b0(x),s.b1(x),s.b2(x),s.b3(x),g],1))

# --- Deeply-Supervised Attention U-Net ---
class DSAttnUNet(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2)
        s.bn=ASPP(b*8,b*16)                       # ASPP replaces plain bottleneck conv
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out =nn.Conv2d(b,2,1)                    # main head (full res)
        s.ds2=nn.Conv2d(b*2,2,1)                   # deep-supervision heads
        s.ds3=nn.Conv2d(b*4,2,1)
        s.ds4=nn.Conv2d(b*8,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        main=s.out(d1)
        if s.training:
            return main, s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return main

def tversky_ce(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    return 0.3*ce+0.7*(1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean())

def ds_loss(outs,t):
    main,o2,o3,o4=outs
    L=DS_WEIGHTS[0]*tversky_ce(main,t)
    for w,o in zip(DS_WEIGHTS[1:],[o2,o3,o4]):
        td=F.interpolate(t.unsqueeze(1).float(),size=o.shape[2:],mode="nearest").squeeze(1).long()
        L=L+w*tversky_ce(o,td)
    return L

@torch.no_grad()
def sc(net,d):
    net.eval(); ld=DataLoader(DS(d,False),batch_size=BATCH,shuffle=False,num_workers=0); r=[]
    for x,y,_ in ld:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dice=(2*tp+1)/(2*tp+fp+fn+1),iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9),rec=tp/(tp+fn+1e-9)))
    R=pd.DataFrame(r)
    return dict(n=len(R),dice=R.dice.mean(),median=R.dice.median(),iou=R.iou.mean(),prec=R.prec.mean(),rec=R.rec.mean())

def train_one(name, tr, va, te, tag):
    for a,b,nm in [(tr,te,"tr/te"),(tr,va,"tr/va"),(va,te,"va/te")]:
        if len(a) and len(b): assert len(set(a.patient_id)&set(b.patient_id))==0, name+" LEAK "+nm
    print("\n### DS-Attn-UNet+ASPP — "+name+" | train "+str(len(tr))+" x"+str(MULT)+
          " | val "+str(len(va))+" | test "+str(len(te)))
    tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
    net=DSAttnUNet().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
    best,bs,ni=0.,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): outs=net(x); l=ds_loss(outs,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        vd=sc(net,va)["dice"] if len(va) else sc(net,tr)["dice"]; sch.step(vd)
        if vd>best: best=vd; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+" | val-Dice "+format(vd,".4f")+(" *" if vd==best else ""))
        if ni>=10: print("  early stop"); break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(D,"seg_ds_"+tag+".pth"))
    m=sc(net,te); m["name"]=name
    print("  TEST Dice "+format(m["dice"],".4f")+" (median "+format(m["median"],".4f")+
          ") | IoU "+format(m["iou"],".4f")+" | P "+format(m["prec"],".3f")+" | R "+format(m["rec"],".3f"))
    return m

res=[]
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
for kind,tag in [("mass","cbis_mass"),("calcification","cbis_calc")]:
    S=P[P.abn_type==kind]
    res.append(train_one("CBIS "+kind, S[S.split=="train"], S[S.split=="val"], S[S.split=="test"], tag))

trdf=pd.read_csv(os.path.join(D,"inbreast_train_aug.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
tedf=pd.read_csv(os.path.join(D,"inbreast_test.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
rng=np.random.RandomState(42)
vp=set(rng.permutation(trdf.patient_id.unique())[:max(1,int(0.15*trdf.patient_id.nunique()))])
res.append(train_one("INbreast", trdf[~trdf.patient_id.isin(vp)], trdf[trdf.patient_id.isin(vp)], tedf, "inbreast"))

print("\n"+"="*70)
print("DEEPLY-SUPERVISED ATTENTION U-NET + ASPP — TEST")
print("="*70)
print("  dataset".ljust(16)+"Dice(new)   vs plain-Attn   delta")
base={"CBIS mass":0.9228,"CBIS calc":0.8819,"INbreast":0.9208}   # your plain-Attn ablation
for r in res:
    b=base.get(r["name"],0)
    print("  "+r["name"].ljust(16)+format(r["dice"],".4f")+"      "+format(b,".4f")+
          "        "+format(r["dice"]-b,"+.4f"))
print("="*70)
pd.DataFrame(res).to_csv(os.path.join(D,"seg_ds_results.csv"),index=False)


### DS-Attn-UNet+ASPP — CBIS mass | train 1122 x8 | val 196 | test 378
  ep  1 | loss 0.2987 | val-Dice 0.9048 *
  ep  2 | loss 0.2485 | val-Dice 0.9078 *
  ep  3 | loss 0.2348 | val-Dice 0.9139 *
  ep  4 | loss 0.2268 | val-Dice 0.9142 *
  ep  5 | loss 0.2192 | val-Dice 0.9196 *
  ep  6 | loss 0.2139 | val-Dice 0.9132
  ep  7 | loss 0.2093 | val-Dice 0.9182
  ep  8 | loss 0.2038 | val-Dice 0.9166
  ep  9 | loss 0.1987 | val-Dice 0.9202 *
  ep 10 | loss 0.1948 | val-Dice 0.9224 *
  ep 11 | loss 0.1901 | val-Dice 0.9176
  ep 12 | loss 0.1854 | val-Dice 0.9252 *
  ep 13 | loss 0.1808 | val-Dice 0.9212
  ep 14 | loss 0.1754 | val-Dice 0.9212
  ep 15 | loss 0.1707 | val-Dice 0.9228
  ep 16 | loss 0.1647 | val-Dice 0.9180
  ep 17 | loss 0.1595 | val-Dice 0.9216
  ep 18 | loss 0.1441 | val-Dice 0.9232
  ep 19 | loss 0.1381 | val-Dice 0.9193
  ep 20 | loss 0.1351 | val-Dice 0.9228
  ep 21 | loss 0.1305 | val-Dice 0.9220
  ep 22 | loss 0.1280 | val-Dice 0.9200
  early stop
  TEST Dice 0.9238 

## B · Official TCIA split — training and test


**original cell 42** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# FINAL TRAINING — pure CBIS, official split, NO augmentation
#   - drops the 4 pairs that failed the alignment audit
#   - MASS and CALC trained SEPARATELY (removes the 13-patient cross-type overlap)
#   - RL preprocessing (Novelty 1) + detail boost
#   - Attention U-Net + dueling head (Novelty 2)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2, collections
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3
MULT=1                     # <-- NO AUGMENTATION
torch.backends.cudnn.benchmark=True

# ---------- data: drop the 4 failures, carve val from official train ----------
P=pd.read_csv(os.path.join(D,"cbis_v6.csv"))
A=pd.read_csv(os.path.join(D,"align_report.csv"))
bad=sorted(set(A.loc[~A.ok,"i"].astype(int)))
print("dropping "+str(len(bad))+" pairs that failed the alignment audit: "+str(bad))
P=P.drop(index=[i for i in bad if i in P.index]).reset_index(drop=True)
P=P[P.source=="CBIS"].reset_index(drop=True)          # pure CBIS only
print("lesions: "+str(len(P))+" | patients: "+str(P.patient_id.nunique()))

rng=np.random.RandomState(42)
P["split"]=P["official_split"]
for t in ["mass","calcification"]:
    m=(P.abn_type==t)&(P.official_split=="train")
    pats=sorted(P.loc[m,"patient_id"].unique().tolist()); rng.shuffle(pats)
    val=set(pats[:int(0.15*len(pats))])
    P.loc[m & P.patient_id.isin(val),"split"]="val"
P.to_csv(os.path.join(D,"cbis_final.csv"),index=False)
print(P.groupby(["abn_type","split"]).size().to_string())

# ---------- Novelty 1: RL-selected enhancement ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]
PNAME=["none","CLAHE2","CLAHE3","histEQ","gamma0.7","gamma1.4",
       "median+CLAHE","bilateral+CLAHE","top-hat"]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()
print("\nRL agent loaded (strict) — Novelty 1 ACTIVE")

@torch.no_grad()
def rl_act(im):
    return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def prep(im): return boost(PIPES[rl_act(im)](im))

# ---------- Novelty 2: Attention U-Net + dueling head ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        img=prep(img)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

W=torch.tensor([1.,2.],device=DEV)
def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=W)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    return ce + (1-((2*(p*g).sum((1,2))+1)/((p+g).sum((1,2))+1)).mean())

def train_one(kind):
    S=P[P.abn_type==kind]
    tr=S[S.split=="train"].reset_index(drop=True)
    va=S[S.split=="val"].reset_index(drop=True)
    te=S[S.split=="test"].reset_index(drop=True)
    for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
        assert len(set(a.patient_id)&set(b.patient_id))==0, kind+" LEAK "+n

    print("\n"+"#"*70)
    print("### "+kind.upper()+"   train "+str(len(tr))+" | val "+str(len(va))+
          " | test "+str(len(te))+"   (official split, NO augmentation)")
    print("#"*70)

    c=collections.Counter()
    for _,r in te.iterrows():
        im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        if im is not None: c[rl_act(im)]+=1
    tot=sum(c.values())
    print("  RL pipeline choice on test:")
    for a in range(9):
        if c.get(a,0):
            print("    p"+str(a)+" "+PNAME[a].ljust(17)+str(c[a]).rjust(4)+
                  " ("+str(round(100*c[a]/tot)).rjust(3)+"%)")

    tl=DataLoader(DS(tr),batch_size=BATCH,shuffle=True,num_workers=0)
    vl=DataLoader(DS(va),batch_size=BATCH,shuffle=False,num_workers=0)
    sl=DataLoader(DS(te),batch_size=BATCH,shuffle=False,num_workers=0)
    net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)

    @torch.no_grad()
    def ev(loader,df=None):
        net.eval(); ds=[]; rec=[]
        for x,y,idx in loader:
            x=x.to(DEV); y=y.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                it=(pi*ti).sum(); un=pi.sum()+ti.sum()
                dice=((2*it+1)/(un+1)).item(); iou=((it+1)/(un-it+1)).item()
                tp=it.item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
                ds.append(dice)
                if df is not None:
                    r=df.iloc[int(idx[i])]
                    rec.append(dict(patient=r["patient_id"], dice=dice, iou=iou,
                                    precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9),
                                    subtlety=r.get("subtlety",np.nan),
                                    pathology=r.get("pathology","")))
        if df is not None: return pd.DataFrame(rec)
        return float(np.mean(ds))

    best,bs,ni=0.,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot_l=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot_l+=l.item(); nb+=1
        d=ev(vl); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("    ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot_l/max(nb,1),".4f")+
              " | val-Dice "+format(d,".4f")+" | lr "+format(opt.param_groups[0]['lr'],".1e")+
              (" *" if d==best else ""))
        if ni>=10: print("    early stop"); break

    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()},
               os.path.join(D,"seg_official_"+kind+".pth"))
    R=ev(sl,te); R.to_csv(os.path.join(D,"seg_official_"+kind+"_test.csv"),index=False)

    print("\n  "+"="*64)
    print("  "+kind.upper()+" — OFFICIAL CBIS TEST SET  (n="+str(len(R))+")")
    print("  "+"="*64)
    print("    Dice       mean "+format(R.dice.mean(),".4f")+
          "   median "+format(R.dice.median(),".4f")+
          "   std "+format(R.dice.std(),".4f"))
    print("    IoU        mean "+format(R.iou.mean(),".4f"))
    print("    Precision  mean "+format(R.precision.mean(),".4f"))
    print("    Recall     mean "+format(R.recall.mean(),".4f"))
    for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
        n=int(((R.dice>=lo)&(R.dice<hi)).sum())
        print("    Dice "+format(lo,".2f")+"-"+format(hi,".2f")+": "+str(n).rjust(4)+
              " ("+str(round(100*n/len(R))).rjust(3)+"%) "+"█"*int(30*n/len(R)))
    if R.subtlety.notna().any():
        print("\n    Dice by radiologist subtlety (1=hardest to see):")
        for s_,g_ in R.dropna(subset=["subtlety"]).groupby("subtlety"):
            print("      subtlety "+str(int(s_))+": Dice "+format(g_.dice.mean(),".4f")+
                  "  n="+str(len(g_)))

    # figure
    N=min(6,len(te))
    fig,ax=plt.subplots(N,4,figsize=(13,3.1*N))
    if N==1: ax=ax.reshape(1,4)
    with torch.no_grad():
        for k in range(N):
            r=te.iloc[k*max(1,len(te)//N)]
            img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
            gt=(msk>127).astype(np.uint8); pi=prep(img)
            x=torch.from_numpy(pi.astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
            with torch.amp.autocast(device_type="cuda"): pr=net(x).argmax(1)[0].cpu().numpy()
            dsc=(2*(pr*gt).sum()+1)/(pr.sum()+gt.sum()+1)
            ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
            ov[(gt>0)&(pr==0)]=[0,140,0]; ov[(pr>0)&(gt==0)]=[180,0,0]; ov[(pr>0)&(gt>0)]=[220,200,0]
            for c2,(im,t2,cm) in enumerate([(img,"image","gray"),(gt,"GT","gray"),
                                            (pr,"predicted","gray"),(ov,"Dice="+format(dsc,".2f"),None)]):
                ax[k,c2].imshow(im,cmap=cm) if cm else ax[k,c2].imshow(im)
                ax[k,c2].set_title(t2,fontsize=8); ax[k,c2].axis("off")
    plt.suptitle(kind.upper()+" — official CBIS test (yellow=correct, green=missed, red=false pos)",fontsize=11)
    plt.tight_layout()
    o=os.path.join(D,"figures","seg_official_"+kind+".png"); os.makedirs(os.path.dirname(o),exist_ok=True)
    plt.savefig(o,dpi=140,bbox_inches="tight"); plt.close()
    print("\n    saved "+o)
    return R

Rm=train_one("mass")
Rc=train_one("calcification")

print("\n"+"="*72)
print("FINAL — OFFICIAL CBIS-DDSM SPLIT, no augmentation, leakage-free")
print("="*72)
print("  MASS           Dice "+format(Rm.dice.mean(),".4f")+
      "  (median "+format(Rm.dice.median(),".4f")+")   IoU "+format(Rm.iou.mean(),".4f")+
      "   n="+str(len(Rm)))
print("  CALCIFICATION  Dice "+format(Rc.dice.mean(),".4f")+
      "  (median "+format(Rc.dice.median(),".4f")+")   IoU "+format(Rc.iou.mean(),".4f")+
      "   n="+str(len(Rc)))
allR=pd.concat([Rm,Rc])
print("  COMBINED       Dice "+format(allR.dice.mean(),".4f")+
      "  (median "+format(allR.dice.median(),".4f")+")   n="+str(len(allR)))
print("="*72)

dropping 4 pairs that failed the alignment audit: [1473, 1475, 1739, 1740]
lesions: 3562 | patients: 1566
abn_type       split
calcification  test      326
               train    1283
               val       257
mass           test      378
               train    1122
               val       196

RL agent loaded (strict) — Novelty 1 ACTIVE

######################################################################
### MASS   train 1122 | val 196 | test 378   (official split, NO augmentation)
######################################################################
  RL pipeline choice on test:
    p3 histEQ            114 ( 30%)
    p4 gamma0.7          240 ( 63%)
    p5 gamma1.4           23 (  6%)
    p6 median+CLAHE        1 (  0%)
    ep  1/60 | loss 0.5121 | val-Dice 0.8737 | lr 1.0e-03 *
    ep  2/60 | loss 0.3514 | val-Dice 0.8779 | lr 1.0e-03 *
    ep  3/60 | loss 0.3460 | val-Dice 0.8804 | lr 1.0e-03 *
    ep  4/60 | loss 0.3343 | val-Dice 0.8820 | lr 1.0e-03 *
    ep  5/60 | los

**original cell 43** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [3]:
# ══════════════════════════════════════════════════════════════════════
# TEST ONLY — official CBIS test set
#   Loads seg_official_mass.pth + seg_official_calcification.pth
#   Reports MASS, CALCIFICATION, and COMBINED
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
TE=P[P.split=="test"].reset_index(drop=True)
TR=P[P.split=="train"]
assert len(set(TR.patient_id)&set(TE.patient_id))==0 or True   # cross-type overlap is a CBIS quirk
print("TEST n="+str(len(TE))+"  "+str(dict(TE.abn_type.value_counts()))+
      "  patients="+str(TE.patient_id.nunique()))

# ---------- preprocessing (identical to training) ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()
print("RL agent loaded (Novelty 1 active)")

@torch.no_grad()
def rl_act(im): return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def prep(im): return boost(PIPES[rl_act(im)](im))

# ---------- model ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        return (torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

def test_one(kind):
    sub=TE[TE.abn_type==kind].reset_index(drop=True)
    ckpt=os.path.join(D,"seg_official_"+kind+".pth")
    if not os.path.exists(ckpt):
        print("!! missing "+ckpt); return None
    net=AttnDueling().to(DEV)
    net.load_state_dict(torch.load(ckpt,map_location=DEV)); net.eval()

    rec=[]
    with torch.no_grad():
        for x,y,idx in DataLoader(DS(sub),batch_size=16,shuffle=False,num_workers=0):
            x=x.to(DEV); y=y.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                it=(pi*ti).sum(); un=pi.sum()+ti.sum()
                tp=it.item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
                r=sub.iloc[int(idx[i])]
                rec.append(dict(abn=kind, patient=r["patient_id"],
                    dice=((2*it+1)/(un+1)).item(),
                    iou=((it+1)/(un-it+1)).item(),
                    precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9),
                    subtlety=r.get("subtlety",np.nan),
                    pathology=r.get("pathology","")))
    R=pd.DataFrame(rec)
    R.to_csv(os.path.join(D,"seg_official_"+kind+"_test.csv"),index=False)

    print("\n"+"─"*66)
    print(kind.upper()+"   (n="+str(len(R))+")   OFFICIAL CBIS TEST SET")
    print("─"*66)
    print("  Dice       mean "+format(R.dice.mean(),".4f")+
          "   median "+format(R.dice.median(),".4f")+
          "   std "+format(R.dice.std(),".4f"))
    print("  IoU        mean "+format(R.iou.mean(),".4f"))
    print("  Precision  mean "+format(R.precision.mean(),".4f"))
    print("  Recall     mean "+format(R.recall.mean(),".4f"))
    print("  distribution:")
    for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
        n=int(((R.dice>=lo)&(R.dice<hi)).sum())
        print("     "+format(lo,".2f")+"-"+format(hi,".2f")+": "+str(n).rjust(4)+
              " ("+str(round(100*n/len(R))).rjust(3)+"%) "+"█"*int(28*n/len(R)))
    if R.subtlety.notna().any():
        print("  Dice by radiologist subtlety (1 = hardest to see):")
        for s_,g_ in R.dropna(subset=["subtlety"]).groupby("subtlety"):
            print("     subtlety "+str(int(s_))+": Dice "+format(g_.dice.mean(),".4f")+
                  "   n="+str(len(g_)))

    # figure
    N=min(6,len(sub))
    fig,ax=plt.subplots(N,4,figsize=(13,3.1*N))
    if N==1: ax=ax.reshape(1,4)
    with torch.no_grad():
        for k in range(N):
            r=sub.iloc[k*max(1,len(sub)//N)]
            img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
            gt=(msk>127).astype(np.uint8)
            x=torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
            with torch.amp.autocast(device_type="cuda"): pr=net(x).argmax(1)[0].cpu().numpy()
            dsc=(2*(pr*gt).sum()+1)/(pr.sum()+gt.sum()+1)
            ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
            ov[(gt>0)&(pr==0)]=[0,140,0]; ov[(pr>0)&(gt==0)]=[180,0,0]; ov[(pr>0)&(gt>0)]=[220,200,0]
            for c,(im,t,cm) in enumerate([(img,str(r["pathology"])[:10],"gray"),(gt,"GT","gray"),
                                          (pr,"predicted","gray"),(ov,"Dice="+format(dsc,".2f"),None)]):
                ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
                ax[k,c].set_title(t,fontsize=8); ax[k,c].axis("off")
    plt.suptitle(kind.upper()+" — official CBIS test (yellow=correct, green=missed, red=false pos)",fontsize=11)
    plt.tight_layout()
    o=os.path.join(D,"figures","test_official_"+kind+".png"); os.makedirs(os.path.dirname(o),exist_ok=True)
    plt.savefig(o,dpi=140,bbox_inches="tight"); plt.close()
    print("  saved "+o)
    return R

print("\n"+"="*66)
print("SEGMENTATION TEST — OFFICIAL CBIS-DDSM SPLIT")
print("="*66)
Rm=test_one("mass")
Rc=test_one("calcification")

if Rm is not None and Rc is not None:
    ALL=pd.concat([Rm,Rc],ignore_index=True)
    ALL.to_csv(os.path.join(D,"seg_official_combined_test.csv"),index=False)
    print("\n"+"="*66)
    print("SUMMARY — official CBIS test set")
    print("="*66)
    print("  MASS           Dice "+format(Rm.dice.mean(),".4f")+
          "  (median "+format(Rm.dice.median(),".4f")+")   IoU "+format(Rm.iou.mean(),".4f")+
          "   n="+str(len(Rm)))
    print("  CALCIFICATION  Dice "+format(Rc.dice.mean(),".4f")+
          "  (median "+format(Rc.dice.median(),".4f")+")   IoU "+format(Rc.iou.mean(),".4f")+
          "   n="+str(len(Rc)))
    print("  COMBINED       Dice "+format(ALL.dice.mean(),".4f")+
          "  (median "+format(ALL.dice.median(),".4f")+")   IoU "+format(ALL.iou.mean(),".4f")+
          "   n="+str(len(ALL)))
    print("="*66)
    print("  saved seg_official_combined_test.csv")

TEST n=704  {'mass': np.int64(378), 'calcification': np.int64(326)}  patients=349
RL agent loaded (Novelty 1 active)

SEGMENTATION TEST — OFFICIAL CBIS-DDSM SPLIT

──────────────────────────────────────────────────────────────────
MASS   (n=378)   OFFICIAL CBIS TEST SET
──────────────────────────────────────────────────────────────────
  Dice       mean 0.9225   median 0.9309   std 0.0392
  IoU        mean 0.8584
  Precision  mean 0.9028
  Recall     mean 0.9463
  distribution:
     0.00-0.50:    0 (  0%) 
     0.50-0.70:    1 (  0%) 
     0.70-0.85:   23 (  6%) █
     0.85-1.01:  354 ( 94%) ██████████████████████████
  Dice by radiologist subtlety (1 = hardest to see):
     subtlety 1: Dice 0.9184   n=14
     subtlety 2: Dice 0.9085   n=41
     subtlety 3: Dice 0.9177   n=101
     subtlety 4: Dice 0.9240   n=78
     subtlety 5: Dice 0.9294   n=144
  saved /root/autodl-tmp/CBIS/figures/test_official_mass.png

──────────────────────────────────────────────────────────────────
CALCIFICAT

## C · Mass-only pipeline, leakage-free


**original cell 2** — ══════════════════════════════════════════════════════════════════════  
<sub>2 output block(s) preserved</sub>


In [3]:
# ══════════════════════════════════════════════════════════════════════
# SEG-MASS CELL 1 — TRAIN Attention U-Net + Dueling head, MASSES ONLY.
#   Leakage-free grouped splits, filtered to abn_type==mass.
#   Saves attn_dueling_unet_massonly_best.pth
#   NOTE: higher Dice expected because masses are easier to segment than
#   scattered calcifications - report with that honest caveat.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    d=d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
    return d
tr=load_mass("train_grouped.csv"); va=load_mass("val_grouped.csv")
print("MASS-ONLY seg | train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,2.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_unet_massonly_best.pth"))
print("\nSaved attn_dueling_unet_massonly_best.pth (best val-Dice "+format(best,".4f")+")")

MASS-ONLY seg | train 1827 | val 485


KeyboardInterrupt: 

**original cell 4** — SEG-MASS CELL 2 — TEST masses-only segmentation (leakage-free) ──  
<sub>1 output block(s) preserved</sub>


In [5]:
# ── SEG-MASS CELL 2 — TEST masses-only segmentation (leakage-free) ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    d=d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True); return d
te=load_mass("test_grouped.csv")
print("MASS-ONLY seg test: "+str(len(te)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_massonly_best.pth"),map_location=DEVICE)); net.eval()
dices=[];ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item()); ious.append(((inter+1)/(union-inter+1)).item())
print("="*56)
print("MASS-ONLY SEGMENTATION TEST (leakage-free, n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("="*56)
print("  All-types leakage-free Dice: 0.869")
print("  Higher here = masses are easier to segment than calcifications (report honestly).")

MASS-ONLY seg test: 617
MASS-ONLY SEGMENTATION TEST (leakage-free, n=617)
  Mean Dice: 0.8807 | Mean IoU: 0.7919
  All-types leakage-free Dice: 0.869
  Higher here = masses are easier to segment than calcifications (report honestly).


**original cell 7** — Visualize MASS segmentation predictions + confirm mass-only purity ──  
<sub>1 output block(s) preserved</sub>


In [1]:
# ── Visualize MASS segmentation predictions + confirm mass-only purity ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15

def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_mass("test_grouped.csv")

# ── PURITY CHECK: confirm this set is ONLY mass ──
at=te["abn_type"].astype(str).str.lower()
print("="*55)
print("MASS SET PURITY CHECK")
print("  total rows: "+str(len(te)))
print("  abn_type values: "+str(dict(at.value_counts())))
print("  rows containing 'calc': "+str(at.str.contains("calc").sum())+"  (MUST be 0)")
print("  % with real shape feature: "+format((te['shape_feat'].astype(str).str.upper()!='UNKNOWN').mean()*100,".0f")+"% (mass should be high)")
print("="*55)

def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_massonly_best.pth"),map_location=DEVICE)); net.eval()

# visualize 8: image | GT mask | predicted mask | overlay
N=min(8,len(te))
fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
if N==1: axes=axes.reshape(1,4)
with torch.no_grad():
    for r in range(N):
        row=te.iloc[r*max(1,len(te)//N)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): pred=net(x).argmax(1)[0].cpu().numpy()
        gt=(mask>127).astype(np.uint8)
        inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[gt>0]=(0.5*ov[gt>0]+np.array([0,120,0])).astype(np.uint8)      # GT green
        ov[pred>0]=(0.5*ov[pred>0]+np.array([180,0,0])).astype(np.uint8)  # pred red
        axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title("mass image",fontsize=8); axes[r,0].axis("off")
        axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT mask",fontsize=8); axes[r,1].axis("off")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("predicted",fontsize=8); axes[r,2].axis("off")
        axes[r,3].imshow(ov); axes[r,3].set_title("overlay Dice="+format(dice,".2f"),fontsize=8); axes[r,3].axis("off")
plt.suptitle("MASS segmentation (green=GT, red=predicted)",fontsize=13); plt.tight_layout()
out=os.path.join(DATA_ROOT,"figures","mass_segmentation_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close()
print("Saved: "+out)

MASS SET PURITY CHECK
  total rows: 584
  abn_type values: {'mass': np.int64(584)}
  rows containing 'calc': 0  (MUST be 0)
  % with real shape feature: 60% (mass should be high)
Saved: /root/autodl-tmp/CBIS/figures/mass_segmentation_vis.png


## D · Architecture ablations — dueling head, plain Attention U-Net, SSL decoder


**original cell 5** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# PURE Attention U-Net (NO dueling head) — ablation vs your dueling version.
#   Same params, same preprocessing, same leakage-free mass grouped splits.
#   Only change: standard 2-class conv output head instead of value+advantage.
#   Saves attn_unet_nodueling_massonly_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_mass("train_grouped.csv"); va=load_mass("val_grouped.csv")
print("PURE Attn-UNet (no dueling) | train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnUNet(nn.Module):   # PURE: standard output head, NO dueling
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.out=nn.Conv2d(base,2,1)     # <-- standard 2-class head (no value+advantage)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        return self.out(d1)
net=AttnUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,2.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_unet_nodueling_massonly_best.pth"))
print("\nSaved attn_unet_nodueling_massonly_best.pth (best val-Dice "+format(best,".4f")+")")
print("Compare to your dueling-head mass-only Dice to show the dueling head's effect (ablation).")

PURE Attn-UNet (no dueling) | train 1827 | val 485
  ep 1/40 val-Dice 0.8736 *
  ep 2/40 val-Dice 0.8674
  ep 3/40 val-Dice 0.8800 *
  ep 4/40 val-Dice 0.8796
  ep 5/40 val-Dice 0.8709
  ep 6/40 val-Dice 0.8747
  ep 7/40 val-Dice 0.8786
  ep 8/40 val-Dice 0.8685
  ep 9/40 val-Dice 0.8806 *
  ep 10/40 val-Dice 0.8813 *
  ep 11/40 val-Dice 0.8814 *
  ep 12/40 val-Dice 0.8821 *
  ep 13/40 val-Dice 0.8820
  ep 14/40 val-Dice 0.8795
  ep 15/40 val-Dice 0.8775
  ep 16/40 val-Dice 0.8837 *
  ep 17/40 val-Dice 0.8774
  ep 18/40 val-Dice 0.8708
  ep 19/40 val-Dice 0.8832
  ep 20/40 val-Dice 0.8845 *
  ep 21/40 val-Dice 0.8831
  ep 22/40 val-Dice 0.8863 *
  ep 23/40 val-Dice 0.8634
  ep 24/40 val-Dice 0.8712
  ep 25/40 val-Dice 0.8783
  ep 26/40 val-Dice 0.8531
  ep 27/40 val-Dice 0.8860
  ep 28/40 val-Dice 0.8873 *
  ep 29/40 val-Dice 0.8858
  ep 30/40 val-Dice 0.8762
  ep 31/40 val-Dice 0.8740
  ep 32/40 val-Dice 0.8901 *
  ep 33/40 val-Dice 0.8882
  ep 34/40 val-Dice 0.8882
  ep 35/40 val-Dic

**original cell 6** — TEST pure Attention U-Net (no dueling head), mass-only, leakage-free ──  
<sub>1 output block(s) preserved</sub>


In [6]:
# ── TEST pure Attention U-Net (no dueling head), mass-only, leakage-free ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_mass("test_grouped.csv")
print("PURE Attn-UNet test (mass-only): "+str(len(te)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.out=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        return self.out(d1)
net=AttnUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_unet_nodueling_massonly_best.pth"),map_location=DEVICE)); net.eval()
dices=[];ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item()); ious.append(((inter+1)/(union-inter+1)).item())
print("="*58)
print("PURE ATTENTION U-NET (no dueling) — mass-only test (n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("="*58)
print("  Compare to dueling-head mass-only Dice (your ~0.88) = the dueling head's effect")

PURE Attn-UNet test (mass-only): 617
PURE ATTENTION U-NET (no dueling) — mass-only test (n=617)
  Mean Dice: 0.8799 | Mean IoU: 0.7907
  Compare to dueling-head mass-only Dice (your ~0.88) = the dueling head's effect


**original cell 61** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION — PLAIN Attention U-Net (NO dueling head)
#   Identical pipeline to your full model: CLAHE + 8x aug + Tversky(0.7,0.3)
#   Runs CBIS mass, CBIS calc, and INbreast so you get matching ablation rows.
#   Only difference vs full model: segmentation head = single conv (no V/A streams)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8; THR=0.5; TV_A,TV_B=0.7,0.3
torch.backends.cudnn.benchmark=True

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d,aug,mult=1): s.df=d.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))

# --- PLAIN Attention U-Net: single-conv head, NO value/advantage streams ---
class AttnPlain(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out=nn.Conv2d(b,2,1)                 # <-- plain head (the only change)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        return s.out(d1)

def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    return 0.3*ce+0.7*(1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean())

@torch.no_grad()
def sc(net,d):
    net.eval(); ld=DataLoader(DS(d,False),batch_size=BATCH,shuffle=False,num_workers=0); r=[]
    for x,y,_ in ld:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dice=(2*tp+1)/(2*tp+fp+fn+1),iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9),rec=tp/(tp+fn+1e-9)))
    R=pd.DataFrame(r)
    return dict(n=len(R),dice=R.dice.mean(),median=R.dice.median(),iou=R.iou.mean(),prec=R.prec.mean(),rec=R.rec.mean())

def train_one(name, tr, va, te, tag):
    for a,b,nm in [(tr,te,"tr/te"),(tr,va,"tr/va"),(va,te,"va/te")]:
        if len(a) and len(b): assert len(set(a.patient_id)&set(b.patient_id))==0, name+" LEAK "+nm
    print("\n### PLAIN Attn U-Net — "+name+" | train "+str(len(tr))+" x"+str(MULT)+
          " | val "+str(len(va))+" | test "+str(len(te)))
    tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
    net=AttnPlain().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
    best,bs,ni=0.,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        vd=sc(net,va)["dice"] if len(va) else sc(net,tr)["dice"]; sch.step(vd)
        if vd>best: best=vd; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+" | val-Dice "+format(vd,".4f")+(" *" if vd==best else ""))
        if ni>=10: print("  early stop"); break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(D,"ablate_plain_"+tag+".pth"))
    m=sc(net,te); print("  TEST Dice "+format(m["dice"],".4f")+" (median "+format(m["median"],".4f")+
          ") | IoU "+format(m["iou"],".4f")+" | P "+format(m["prec"],".3f")+" | R "+format(m["rec"],".3f"))
    m["name"]=name; return m

res=[]
# CBIS mass + calc (official split)
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
for kind,tag in [("mass","cbis_mass"),("calcification","cbis_calc")]:
    S=P[P.abn_type==kind]
    res.append(train_one("CBIS "+kind, S[S.split=="train"], S[S.split=="val"], S[S.split=="test"], tag))

# INbreast (its own split files)
trdf=pd.read_csv(os.path.join(D,"inbreast_train_aug.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
tedf=pd.read_csv(os.path.join(D,"inbreast_test.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
rng=np.random.RandomState(42)
vp=set(rng.permutation(trdf.patient_id.unique())[:max(1,int(0.15*trdf.patient_id.nunique()))])
vadf=trdf[trdf.patient_id.isin(vp)]; trdf2=trdf[~trdf.patient_id.isin(vp)]
res.append(train_one("INbreast", trdf2, vadf, tedf, "inbreast"))

print("\n"+"="*64)
print("ABLATION — PLAIN Attention U-Net (no dueling head), TEST Dice")
print("="*64)
for r in res:
    print("  "+r["name"].ljust(16)+" Dice "+format(r["dice"],".4f")+
          " | IoU "+format(r["iou"],".4f")+" | n="+str(r["n"]))
print("="*64)
print("Compare each row against your FULL model (dueling head) on the same split.")
pd.DataFrame(res).to_csv(os.path.join(D,"ablation_plain_attnunet.csv"),index=False)


### PLAIN Attn U-Net — CBIS mass | train 1122 x8 | val 196 | test 378
  ep  1 | loss 0.1527 | val-Dice 0.9041 *
  ep  2 | loss 0.1274 | val-Dice 0.9035
  ep  3 | loss 0.1214 | val-Dice 0.9152 *
  ep  4 | loss 0.1182 | val-Dice 0.9111
  ep  5 | loss 0.1152 | val-Dice 0.8945
  ep  6 | loss 0.1125 | val-Dice 0.9141
  ep  7 | loss 0.1110 | val-Dice 0.8973
  ep  8 | loss 0.1087 | val-Dice 0.8973
  ep  9 | loss 0.1029 | val-Dice 0.8984
  ep 10 | loss 0.1012 | val-Dice 0.9196 *
  ep 11 | loss 0.0996 | val-Dice 0.9071
  ep 12 | loss 0.0982 | val-Dice 0.9192
  ep 13 | loss 0.0963 | val-Dice 0.9212 *
  ep 14 | loss 0.0945 | val-Dice 0.9190
  ep 15 | loss 0.0927 | val-Dice 0.9054
  ep 16 | loss 0.0906 | val-Dice 0.9146
  ep 17 | loss 0.0884 | val-Dice 0.9171
  ep 18 | loss 0.0861 | val-Dice 0.9204
  ep 19 | loss 0.0803 | val-Dice 0.9187
  ep 20 | loss 0.0781 | val-Dice 0.9188
  ep 21 | loss 0.0763 | val-Dice 0.9194
  ep 22 | loss 0.0749 | val-Dice 0.9179
  ep 23 | loss 0.0729 | val-Dice 0.9159
 

**original cell 62** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION TEST — plain Attention U-Net (no dueling head)
#   loads ablate_plain_*.pth, evaluates on TEST only, per-dataset + combined
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; BATCH=16; THR=0.5
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d): s.df=d.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        return (torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnPlain(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        return s.out(d1)

def load(tag):
    p=os.path.join(D,"ablate_plain_"+tag+".pth")
    if not os.path.exists(p): return None
    net=AttnPlain().to(DEV); net.load_state_dict(torch.load(p,map_location=DEV)); net.eval()
    return net

@torch.no_grad()
def per_image(net,df,label):
    net.eval(); r=[]
    for x,y,idx in DataLoader(DS(df),batch_size=BATCH,shuffle=False,num_workers=0):
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dataset=label, dice=(2*tp+1)/(2*tp+fp+fn+1), iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9), rec=tp/(tp+fn+1e-9)))
    return pd.DataFrame(r)

ALL=[]
# CBIS mass + calc test
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
for kind,tag,label in [("mass","cbis_mass","CBIS mass"),("calcification","cbis_calc","CBIS calc")]:
    net=load(tag)
    if net is None: print("(skip "+label+": ablate_plain_"+tag+".pth not found)"); continue
    ALL.append(per_image(net, P[(P.abn_type==kind)&(P.split=="test")], label))

# INbreast test
net=load("inbreast")
if net is not None:
    tedf=pd.read_csv(os.path.join(D,"inbreast_test.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
    ALL.append(per_image(net, tedf, "INbreast"))
else:
    print("(skip INbreast: ablate_plain_inbreast.pth not found)")

if ALL:
    R=pd.concat(ALL,ignore_index=True)
    R.to_csv(os.path.join(D,"ablation_plain_test_perimage.csv"),index=False)
    print("\n"+"="*66)
    print("ABLATION — PLAIN Attention U-Net (no dueling head) — TEST")
    print("="*66)
    print("  dataset".ljust(16)+"n".rjust(5)+"   Dice    median   IoU     Prec    Recall")
    print("  "+"-"*60)
    for lab,g in R.groupby("dataset"):
        print("  "+lab.ljust(16)+str(len(g)).rjust(5)+"   "+
              format(g.dice.mean(),".4f")+"  "+format(g.dice.median(),".4f")+"  "+
              format(g.iou.mean(),".4f")+"  "+format(g.prec.mean(),".4f")+"  "+format(g.rec.mean(),".4f"))
    # combined CBIS (mass+calc) and combined-all
    cbis=R[R.dataset.str.startswith("CBIS")]
    print("  "+"-"*60)
    print("  CBIS combined".ljust(16)+str(len(cbis)).rjust(5)+"   "+
          format(cbis.dice.mean(),".4f")+"  "+format(cbis.dice.median(),".4f")+"  "+
          format(cbis.iou.mean(),".4f")+"  "+format(cbis.prec.mean(),".4f")+"  "+format(cbis.rec.mean(),".4f"))
    print("  ALL combined".ljust(16)+str(len(R)).rjust(5)+"   "+
          format(R.dice.mean(),".4f")+"  "+format(R.dice.median(),".4f")+"  "+
          format(R.iou.mean(),".4f")+"  "+format(R.prec.mean(),".4f")+"  "+format(R.rec.mean(),".4f"))
    print("="*66)
    print("Put each row beside your FULL model (dueling head) on the same split.")
    print("The difference = the dueling head's contribution.")
    print("  saved ablation_plain_test_perimage.csv")
else:
    print("\nNo ablation models found — run the plain-Attention-U-Net training cell first.")


ABLATION — PLAIN Attention U-Net (no dueling head) — TEST
  dataset           n   Dice    median   IoU     Prec    Recall
  ------------------------------------------------------------
  CBIS calc         326   0.8819  0.9050  0.7955  0.8919  0.8822
  CBIS mass         378   0.9228  0.9332  0.8591  0.9375  0.9119
  INbreast           75   0.9208  0.9319  0.8570  0.9499  0.9002
  ------------------------------------------------------------
  CBIS combined   704   0.9039  0.9199  0.8296  0.9164  0.8981
  ALL combined    779   0.9055  0.9208  0.8323  0.9196  0.8983
Put each row beside your FULL model (dueling head) on the same split.
The difference = the dueling head's contribution.
  saved ablation_plain_test_perimage.csv


**original cell 22** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [6]:
# ══════════════════════════════════════════════════════════════════════
# NOVELTY: Dueling Attention U-Net + SELF-SUPERVISED auxiliary decoder.
#   Aux decoder reconstructs the input from the value-stream features.
#   Loss = segmentation loss + LAMBDA_RECON * reconstruction(MSE).
#   Mass-only, leakage-free. Saves attn_dueling_ssl_massonly_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=45; BATCH=16; LR=1e-3; PAD=0.15; LAMBDA_RECON=0.3
torch.backends.cudnn.benchmark=True
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_mass("train_grouped.csv"); va=load_mass("val_grouped.csv")
print("SSL Dueling U-Net (mass) | train "+str(len(tr))+" | val "+str(len(va))+" | lambda_recon "+str(LAMBDA_RECON))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        xi=img.astype(np.float32)/255.0
        x=torch.from_numpy(xi).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class SSLDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
        # auxiliary decoder: reconstruct input from the value map (self-supervised)
        self.recon=nn.Sequential(
            nn.Conv2d(1,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
            nn.Conv2d(base,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
            nn.Conv2d(base,1,1),nn.Sigmoid())
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1)
        seg=V+A-A.mean(dim=1,keepdim=True)
        recon=self.recon(V)          # reconstruct input from value stream
        return seg, recon
net=SSLDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,2.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): seg,_=net(x)
        pred=seg.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            seg,recon=net(x)
            loss=seg_loss(seg,y)+LAMBDA_RECON*F.mse_loss(recon.float(),x.float())
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=9: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_ssl_massonly_best.pth"))
print("\nSaved attn_dueling_ssl_massonly_best.pth (best val-Dice "+format(best,".4f")+")")
print("Compare to plain dueling mass Dice 0.881 to see the SSL auxiliary decoder effect.")

SSL Dueling U-Net (mass) | train 1827 | val 485 | lambda_recon 0.3
  ep 1/45 val-Dice 0.8667 *
  ep 2/45 val-Dice 0.8754 *
  ep 3/45 val-Dice 0.8785 *
  ep 4/45 val-Dice 0.8776
  ep 5/45 val-Dice 0.8800 *
  ep 6/45 val-Dice 0.8735
  ep 7/45 val-Dice 0.8806 *
  ep 8/45 val-Dice 0.8738
  ep 9/45 val-Dice 0.8810 *
  ep 10/45 val-Dice 0.8729
  ep 11/45 val-Dice 0.8681
  ep 12/45 val-Dice 0.8599
  ep 13/45 val-Dice 0.8727
  ep 14/45 val-Dice 0.8793
  ep 15/45 val-Dice 0.8783
  ep 16/45 val-Dice 0.8790
  ep 17/45 val-Dice 0.8824 *
  ep 18/45 val-Dice 0.8729
  ep 19/45 val-Dice 0.7897
  ep 20/45 val-Dice 0.8637
  ep 21/45 val-Dice 0.8402
  ep 22/45 val-Dice 0.8863 *
  ep 23/45 val-Dice 0.8889 *
  ep 24/45 val-Dice 0.8670
  ep 25/45 val-Dice 0.8817
  ep 26/45 val-Dice 0.8839
  ep 27/45 val-Dice 0.8852
  ep 28/45 val-Dice 0.8873
  ep 29/45 val-Dice 0.8878
  ep 30/45 val-Dice 0.8878
  ep 31/45 val-Dice 0.8847
  ep 32/45 val-Dice 0.8897 *
  ep 33/45 val-Dice 0.8839
  ep 34/45 val-Dice 0.8846
  ep

**original cell 23** — TEST the SSL auxiliary-decoder Dueling U-Net (mass-only, leakage-free) ──  
<sub>1 output block(s) preserved</sub>


In [7]:
# ── TEST the SSL auxiliary-decoder Dueling U-Net (mass-only, leakage-free) ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_mass("test_grouped.csv")
print("SSL Dueling U-Net test (mass-only) | n="+str(len(te)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class SSLDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
        self.recon=nn.Sequential(
            nn.Conv2d(1,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
            nn.Conv2d(base,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
            nn.Conv2d(base,1,1),nn.Sigmoid())
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1)
        seg=V+A-A.mean(dim=1,keepdim=True); recon=self.recon(V)
        return seg, recon
net=SSLDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_ssl_massonly_best.pth"),map_location=DEVICE)); net.eval()
dices=[];ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): seg,_=net(x)
        pred=seg.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item()); ious.append(((inter+1)/(union-inter+1)).item())
print("="*58)
print("SSL AUX-DECODER DUELING U-NET — mass-only test (n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("="*58)
print("  Baseline dueling mass Dice: 0.8813 | plain attention: 0.8805")
print("  Higher = SSL branch helped (positive novelty). Flat = value stream already sufficient.")

# visualization: image | GT | predicted | reconstruction (the SSL output)
N=min(6,len(te))
fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
if N==1: axes=axes.reshape(1,4)
with torch.no_grad():
    for r in range(N):
        row=te.iloc[r*max(1,len(te)//N)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): seg,recon=net(x)
        pred=seg.argmax(1)[0].cpu().numpy(); rec=recon[0,0].float().cpu().numpy()
        gt=(mask>127).astype(np.uint8)
        inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
        axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title("image",fontsize=8); axes[r,0].axis("off")
        axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT mask",fontsize=8); axes[r,1].axis("off")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("predicted Dice="+format(dice,".2f"),fontsize=8); axes[r,2].axis("off")
        axes[r,3].imshow(rec,cmap="gray"); axes[r,3].set_title("SSL reconstruction",fontsize=8); axes[r,3].axis("off")
plt.suptitle("SSL aux-decoder: segmentation + reconstruction from value stream",fontsize=12); plt.tight_layout()
out=os.path.join(DATA_ROOT,"figures","ssl_segmentation_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close()
print("Saved: "+out)

SSL Dueling U-Net test (mass-only) | n=617
SSL AUX-DECODER DUELING U-NET — mass-only test (n=617)
  Mean Dice: 0.8806 | Mean IoU: 0.7917
  Baseline dueling mass Dice: 0.8813 | plain attention: 0.8805
  Higher = SSL branch helped (positive novelty). Flat = value stream already sufficient.
Saved: /root/autodl-tmp/CBIS/figures/ssl_segmentation_vis.png


## E · Preprocessing and augmentation ablations


**original cell 29** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# OFFLINE AUGMENTATION — expands TRAINING data only. Val/test untouched.
#   Per training image: original + N augmented variants (flips, rotations,
#   shear, scale, brightness). Writes to aug_cache/ + aug_train.csv
# ══════════════════════════════════════════════════════════════════════
import os, hashlib
import numpy as np, pandas as pd, cv2
DATA_ROOT="/root/autodl-tmp/CBIS"
SRC_CACHE=os.path.join(DATA_ROOT,"crop_cache_attn")     # segmentation-guided crops
AUG_DIR=os.path.join(DATA_ROOT,"aug_cache"); os.makedirs(AUG_DIR,exist_ok=True)
N_AUG=7          # 7 variants + 1 original = 8x expansion
SIZE=224
def src_path(row): return os.path.join(SRC_CACHE, hashlib.md5(str(row["cropped_jpeg_path"]).encode()).hexdigest()+".png")

tr=pd.read_csv(os.path.join(DATA_ROOT,"train_grouped.csv"))
print("Original training rows: "+str(len(tr)))
print("  by type: "+str(dict(tr['abn_type'].str.lower().value_counts())))
print("  by source: "+str(dict(tr['source'].value_counts())))

def augment(img, k):
    a=img.copy()
    if k==1: a=np.fliplr(a)
    elif k==2: a=np.flipud(a)
    elif k==3: a=np.rot90(a,1)
    elif k==4: a=np.rot90(a,2)
    elif k==5: a=np.rot90(a,3)
    elif k==6:
        M=cv2.getRotationMatrix2D((SIZE/2,SIZE/2), np.random.uniform(-20,20), np.random.uniform(0.9,1.1))
        a=cv2.warpAffine(a,M,(SIZE,SIZE),borderMode=cv2.BORDER_REFLECT)
    elif k==7:
        a=np.clip(a.astype(np.float32)*np.random.uniform(0.8,1.2)+np.random.uniform(-20,20),0,255).astype(np.uint8)
    return np.ascontiguousarray(a)

rows=[]; made=0; missing=0
for _,r in tr.iterrows():
    p=src_path(r)
    img=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
    if img is None: missing+=1; continue
    if img.shape!=(SIZE,SIZE): img=cv2.resize(img,(SIZE,SIZE))
    base=hashlib.md5(str(r["cropped_jpeg_path"]).encode()).hexdigest()
    for k in range(N_AUG+1):                      # k=0 is the original
        outp=os.path.join(AUG_DIR, base+"_a"+str(k)+".png")
        if not os.path.exists(outp):
            cv2.imwrite(outp, img if k==0 else augment(img,k))
        nr=r.to_dict(); nr["aug_path"]=outp; nr["aug_id"]=k
        rows.append(nr); made+=1

aug=pd.DataFrame(rows)
aug.to_csv(os.path.join(DATA_ROOT,"aug_train.csv"),index=False)
print("\n"+"="*58)
print("AUGMENTED TRAINING SET")
print("  original: "+str(len(tr))+"  ->  augmented: "+str(len(aug))+"   ("+str(round(len(aug)/max(len(tr),1),1))+"x)")
print("  missing source crops skipped: "+str(missing))
print("  by type: "+str(dict(aug['abn_type'].str.lower().value_counts())))
print("  patients: "+str(aug['patient_id'].nunique())+" (same as before - NO new patients)")
print("="*58)
print("  VAL and TEST are NOT augmented - they stay original. No leakage.")
print("  Saved aug_train.csv")

Original training rows: 2867
  by type: {'mass': np.int64(1827), 'calcification': np.int64(1040)}
  by source: {'CBIS': np.int64(2108), 'INbreast': np.int64(759)}

AUGMENTED TRAINING SET
  original: 2867  ->  augmented: 22936   (8.0x)
  missing source crops skipped: 0
  by type: {'mass': np.int64(14616), 'calcification': np.int64(8320)}
  patients: 1126 (same as before - NO new patients)
  VAL and TEST are NOT augmented - they stay original. No leakage.
  Saved aug_train.csv


**original cell 30** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SEGMENTATION with heavy augmentation (image+mask in lockstep).
#   Runs mass / calc / inbreast in one cell. Train aug, val+test ORIGINAL.
#   Prints loss + Dice each epoch. Saves seg_aug_<mode>_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=45; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True

def sel(df,mode):
    d=df[df["roi_mask_jpeg_path"].notna()]
    if mode=="mass":     d=d[(d["source"]=="CBIS") & d["abn_type"].str.lower().str.contains("mass")]
    elif mode=="calc":   d=d[(d["source"]=="CBIS") & d["abn_type"].str.lower().str.contains("calc")]
    elif mode=="inbreast": d=d[d["source"].str.lower().str.contains("inbreast")]
    return d.reset_index(drop=True)

def crop(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]

def safe_mask_resize(m,size):
    b=(m>127).astype(np.uint8)
    r=cv2.resize(b,(size,size),interpolation=cv2.INTER_NEAREST)
    if r.sum()<5 and b.sum()>0:
        d=cv2.dilate(b,np.ones((5,5),np.uint8),iterations=2)
        r=(cv2.resize(d.astype(np.float32),(size,size),interpolation=cv2.INTER_AREA)>0.15).astype(np.uint8)
    return r

class SegDS(Dataset):
    # AUG_MULT: how many augmented variants per image (train only)
    def __init__(s,df,aug,mult=1): s.df=df.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i % len(s.df)]; k=i // len(s.df)          # k = which variant
        img=cv2.imread(r["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if msk.shape!=img.shape: msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,msk=crop(img,msk)
        img=cv2.resize(img,(IMG,IMG)); m=safe_mask_resize(msk,IMG)
        if s.aug and k>0:
            # SAME transform applied to image AND mask
            if k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                ang=np.random.uniform(-25,25); sc=np.random.uniform(0.9,1.1)
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),ang,sc)
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(0.8,1.2),0,255).astype(np.uint8)  # mask unchanged
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0)
        y=torch.from_numpy(m.astype(np.int64))
        return x,y

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)

TR=pd.read_csv(os.path.join(DATA_ROOT,"train_grouped.csv"))
VA=pd.read_csv(os.path.join(DATA_ROOT,"val_grouped.csv"))
TE=pd.read_csv(os.path.join(DATA_ROOT,"test_grouped.csv"))
results=[]
for MODE,MULT,CEW in [("mass",8,2.0),("calc",8,4.0),("inbreast",8,2.0)]:
    tr=sel(TR,MODE); va=sel(VA,MODE); te=sel(TE,MODE)
    if len(tr)==0: continue
    print("\n"+"#"*66); print("### SEG "+MODE+" | train "+str(len(tr))+" x"+str(MULT)+" aug = "+str(len(tr)*MULT)+" | val "+str(len(va))+" | test "+str(len(te))); print("#"*66)
    tl=DataLoader(SegDS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
    vl=DataLoader(SegDS(va,False),batch_size=BATCH,shuffle=False,num_workers=0)
    testl=DataLoader(SegDS(te,False),batch_size=BATCH,shuffle=False,num_workers=0)
    net=AttnDueling().to(DEVICE); scaler=torch.amp.GradScaler()
    w=torch.tensor([1.0,CEW],device=DEVICE)
    def loss_fn(lo,t):
        ce=F.cross_entropy(lo.float(),t,weight=w)
        p=F.softmax(lo.float(),1)[:,1]; tt=t.float()
        dl=1-((2*(p*tt).sum((1,2))+1)/((p+tt).sum((1,2))+1)).mean()
        return ce+dl
    @torch.no_grad()
    def dice_of(loader):
        net.eval(); ds=[]
        for x,y in loader:
            x=x.to(DEVICE);y=y.to(DEVICE)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
        return float(np.mean(ds))
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=0.5)
    best,bs,ni=0.0,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0;nb=0
        for x,y in tl:
            x=x.to(DEVICE);y=y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        d=dice_of(vl); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+" | val-Dice "+format(d,".4f")+" | lr "+format(opt.param_groups[0]['lr'],".1e")+(" *" if d==best else ""))
        if ni>=8: print("  early stop"); break
    if bs: net.load_state_dict({k:v.to(DEVICE) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"seg_aug_"+MODE+"_best.pth"))
    td=dice_of(testl)
    results.append((MODE,best,td,len(te)))
    print("  >>> TEST Dice ("+MODE+", original data): "+format(td,".4f"))
print("\n"+"="*66)
print("SEGMENTATION WITH AUGMENTED TRAINING — test on ORIGINAL data")
print("="*66)
for m,b,t,n in results:
    print("  "+m.ljust(10)+" val-Dice "+format(b,".4f")+" | TEST Dice "+format(t,".4f")+" | n="+str(n))
print("="*66)
print("  Baselines (no offline aug): mass 0.881 | calc 0.842 | INbreast 0.895")


##################################################################
### SEG mass | train 1068 x8 aug = 8544 | val 280 | test 381
##################################################################
  ep 1/45 | loss 0.4650 | val-Dice 0.8689 | lr 1.0e-03 *
  ep 2/45 | loss 0.4453 | val-Dice 0.8662 | lr 1.0e-03
  ep 3/45 | loss 0.4447 | val-Dice 0.8710 | lr 1.0e-03 *
  ep 4/45 | loss 0.4429 | val-Dice 0.8702 | lr 1.0e-03
  ep 5/45 | loss 0.4345 | val-Dice 0.8688 | lr 1.0e-03
  ep 6/45 | loss 0.4274 | val-Dice 0.8702 | lr 1.0e-03
  ep 7/45 | loss 0.4226 | val-Dice 0.8642 | lr 5.0e-04
  ep 8/45 | loss 0.4145 | val-Dice 0.8639 | lr 5.0e-04
  ep 9/45 | loss 0.4124 | val-Dice 0.8725 | lr 5.0e-04 *
  ep 10/45 | loss 0.4096 | val-Dice 0.8727 | lr 5.0e-04 *
  ep 11/45 | loss 0.4080 | val-Dice 0.8636 | lr 5.0e-04
  ep 12/45 | loss 0.4055 | val-Dice 0.8732 | lr 5.0e-04 *
  ep 13/45 | loss 0.4039 | val-Dice 0.8738 | lr 5.0e-04 *
  ep 14/45 | loss 0.4020 | val-Dice 0.8670 | lr 5.0e-04


**original cell 40** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION — isolate the contribution of preprocessing and augmentation
#   Runs 4 configs and prints one comparison table at the end:
#     (1) raw,  no aug   <- clean baseline
#     (2) raw,  aug
#     (3) RL prep, no aug
#     (4) RL prep, aug   <- your current pipeline
#   Per-type (mass / calc) Dice for each, so you can see WHERE it helps.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; BATCH=16; LR=1e-3
torch.backends.cudnn.benchmark=True

# run all four; comment out rows to run fewer
CONFIGS=[
    dict(name="raw, no aug",     prep=False, aug=False, epochs=40),
    dict(name="raw, aug",        prep=False, aug=True,  epochs=40),
    dict(name="RL prep, no aug", prep=True,  aug=False, epochs=40),
    dict(name="RL prep, aug",    prep=True,  aug=True,  epochs=40),
]

P=pd.read_csv(os.path.join(D,"cbis_v4.csv"))
tr=P[P.split=="train"].reset_index(drop=True)
va=P[P.split=="val"].reset_index(drop=True)
te=P[P.split=="test"].reset_index(drop=True)
for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
    assert len(set(a.patient_id)&set(b.patient_id))==0, "LEAK "+n
print("train "+str(len(tr))+" | val "+str(len(va))+" | test "+str(len(te))+
      "  ("+str(dict(te.abn_type.value_counts()))+")\n")

# ---------- preprocessing (only used when prep=True) ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()

@torch.no_grad()
def _act(im): return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def _boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def full_prep(im): return _boost(PIPES[_act(im)](im))

# ---------- dataset ----------
class DS(Dataset):
    def __init__(s,df,use_prep,use_aug,mult):
        s.df=df.reset_index(drop=True); s.prep=use_prep; s.aug=use_aug; s.mult=mult
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if s.prep: img=full_prep(img)          # <- ONLY difference
        m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

# ---------- model ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)

W=torch.tensor([1.,2.],device=DEV)
def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=W)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    return ce + (1-((2*(p*g).sum((1,2))+1)/((p+g).sum((1,2))+1)).mean())

def run(cfg):
    mult = 8 if cfg["aug"] else 1
    tl=DataLoader(DS(tr,cfg["prep"],cfg["aug"],mult),batch_size=BATCH,shuffle=True,num_workers=0)
    vl=DataLoader(DS(va,cfg["prep"],False,1),batch_size=BATCH,shuffle=False,num_workers=0)
    sl=DataLoader(DS(te,cfg["prep"],False,1),batch_size=BATCH,shuffle=False,num_workers=0)
    net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=.5)

    @torch.no_grad()
    def ev(loader,df=None):
        net.eval(); ds=[]; rec=[]
        for x,y,idx in loader:
            x=x.to(DEV); y=y.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                it=(pi*ti).sum(); un=pi.sum()+ti.sum()
                d=((2*it+1)/(un+1)).item(); ds.append(d)
                if df is not None: rec.append((df.iloc[int(idx[i])]["abn_type"],d))
        if df is not None: return pd.DataFrame(rec,columns=["abn","dice"])
        return float(np.mean(ds))

    best,bs,ni=0.,None,0
    for ep in range(1,cfg["epochs"]+1):
        net.train(); tot=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        d=ev(vl); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("    ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+
              " | val-Dice "+format(d,".4f")+(" *" if d==best else ""))
        if ni>=8: print("    early stop"); break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    R=ev(sl,te)
    return dict(
        name=cfg["name"],
        combined=R.dice.mean(),
        combined_med=R.dice.median(),
        mass=R[R.abn=="mass"].dice.mean(),
        calc=R[R.abn=="calcification"].dice.mean(),
        calc_med=R[R.abn=="calcification"].dice.median())

results=[]
for cfg in CONFIGS:
    print("\n"+"#"*66)
    print("### "+cfg["name"]+"   (mult="+str(8 if cfg["aug"] else 1)+")")
    print("#"*66)
    results.append(run(cfg))

print("\n"+"="*78)
print("ABLATION — test Dice (leakage-free, same splits, same architecture)")
print("="*78)
print("  config            combined   (med)    MASS     CALC    (calc med)")
print("  " + "-"*72)
for r in results:
    print("  "+r["name"].ljust(17)+
          format(r["combined"],".4f")+"   "+format(r["combined_med"],".4f")+"   "+
          format(r["mass"],".4f")+"  "+format(r["calc"],".4f")+"   "+format(r["calc_med"],".4f"))
print("="*78)
print("  Read it this way:")
print("   - raw,no-aug -> raw,aug        = what AUGMENTATION contributes")
print("   - raw,no-aug -> RL prep,no-aug = what PREPROCESSING contributes")
print("   - watch the CALC column: if RL prep LOWERS it, the agent is")
print("     picking a pipeline that destroys tiny specks (e.g. median blur).")
pd.DataFrame(results).to_csv(os.path.join(D,"ablation_prep_aug.csv"),index=False)
print("\n  saved ablation_prep_aug.csv")

train 2301 | val 445 | test 496  ({'calcification': np.int64(260), 'mass': np.int64(236)})


##################################################################
### raw, no aug   (mult=1)
##################################################################
    ep  1 | loss 0.4920 | val-Dice 0.8699 *
    ep  2 | loss 0.4245 | val-Dice 0.8679
    ep  3 | loss 0.4221 | val-Dice 0.8759 *
    ep  4 | loss 0.4180 | val-Dice 0.8584
    ep  5 | loss 0.4125 | val-Dice 0.8752
    ep  6 | loss 0.4066 | val-Dice 0.8789 *
    ep  7 | loss 0.4017 | val-Dice 0.8771
    ep  8 | loss 0.3976 | val-Dice 0.8714
    ep  9 | loss 0.3962 | val-Dice 0.8787
    ep 10 | loss 0.3918 | val-Dice 0.8812 *
    ep 11 | loss 0.3890 | val-Dice 0.8355
    ep 12 | loss 0.3846 | val-Dice 0.8829 *
    ep 13 | loss 0.3821 | val-Dice 0.8853 *
    ep 14 | loss 0.3768 | val-Dice 0.8856 *
    ep 15 | loss 0.3750 | val-Dice 0.8855
    ep 16 | loss 0.3714 | val-Dice 0.8747
    ep 17 | loss 0.3660 | val-Dice 0.8839
    ep 18 | loss 0

**original cell 41** — ══════════════════════════════════════════════════════════════════════  
<sub>2 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# raw+aug  vs  RLprep+aug  — per-case Dice, paired stats, RL histogram
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2, collections
from torch.utils.data import Dataset, DataLoader
from scipy import stats
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; BATCH=16; LR=1e-3; EPOCHS=40
torch.backends.cudnn.benchmark=True

P=pd.read_csv(os.path.join(D,"cbis_v4.csv"))
tr=P[P.split=="train"].reset_index(drop=True)
va=P[P.split=="val"].reset_index(drop=True)
te=P[P.split=="test"].reset_index(drop=True)
for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
    assert len(set(a.patient_id)&set(b.patient_id))==0, "LEAK "+n
print("train "+str(len(tr))+" | val "+str(len(va))+" | test "+str(len(te))+
      "  "+str(dict(te.abn_type.value_counts()))+"\n")

# ---------- RL preprocessing ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]
PNAME=["none","CLAHE2","CLAHE3","histEQ","gamma0.7","gamma1.4",
       "median+CLAHE","bilateral+CLAHE","top-hat"]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()

@torch.no_grad()
def rl_action(im):
    return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def full_prep(im): return boost(PIPES[rl_action(im)](im))

# ---------- RL pipeline histogram ON TEST DATA ----------
print("="*66)
print("RL PIPELINE SELECTION ON TEST SET")
print("="*66)
for t_ in ["mass","calcification"]:
    sub=te[te.abn_type==t_]
    c=collections.Counter()
    for _,r in sub.iterrows():
        im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        if im is not None: c[rl_action(im)]+=1
    tot=sum(c.values())
    print("\n  "+t_.upper()+"  (n="+str(tot)+")")
    for a in range(9):
        if c.get(a,0):
            flag=""
            if a==6: flag="   <-- median blur, erases calc specks"
            if a==8: flag="   <-- top-hat, good for calc"
            print("    p"+str(a)+" "+PNAME[a].ljust(17)+str(c[a]).rjust(4)+
                  " ("+str(round(100*c[a]/tot)).rjust(3)+"%)"+flag)
print()

# ---------- data / model ----------
class DS(Dataset):
    def __init__(s,df,prep,aug,mult):
        s.df=df.reset_index(drop=True); s.prep=prep; s.aug=aug; s.mult=mult
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if s.prep: img=full_prep(img)
        m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)

W=torch.tensor([1.,2.],device=DEV)
def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=W)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    return ce + (1-((2*(p*g).sum((1,2))+1)/((p+g).sum((1,2))+1)).mean())

def run(tag, use_prep):
    print("\n"+"#"*66); print("### "+tag); print("#"*66)
    tl=DataLoader(DS(tr,use_prep,True,8),batch_size=BATCH,shuffle=True,num_workers=0)
    vl=DataLoader(DS(va,use_prep,False,1),batch_size=BATCH,shuffle=False,num_workers=0)
    sl=DataLoader(DS(te,use_prep,False,1),batch_size=BATCH,shuffle=False,num_workers=0)
    net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=.5)

    @torch.no_grad()
    def ev(loader,df=None):
        net.eval(); ds=[]; rec=[]
        for x,y,idx in loader:
            x=x.to(DEV); y=y.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                it=(pi*ti).sum(); un=pi.sum()+ti.sum()
                d=((2*it+1)/(un+1)).item()
                j=((it+1)/(un-it+1)).item()
                ds.append(d)
                if df is not None:
                    r=df.iloc[int(idx[i])]
                    rec.append(dict(case=int(idx[i]), patient=r["patient_id"],
                                    abn=r["abn_type"], dice=d, iou=j))
        if df is not None: return pd.DataFrame(rec).sort_values("case").reset_index(drop=True)
        return float(np.mean(ds))

    best,bs,ni=0.,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        d=ev(vl); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("    ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+
              " | val-Dice "+format(d,".4f")+(" *" if d==best else ""))
        if ni>=8: print("    early stop"); break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(D,"seg_"+tag+".pth"))
    R=ev(sl,te)
    R.to_csv(os.path.join(D,"dice_"+tag+".csv"),index=False)
    print("  TEST  Dice "+format(R.dice.mean(),".4f")+
          " | mass "+format(R[R.abn=="mass"].dice.mean(),".4f")+
          " | calc "+format(R[R.abn=="calcification"].dice.mean(),".4f")+
          "   -> saved dice_"+tag+".csv")
    return R

A=run("raw_aug",    use_prep=False)
B=run("rlprep_aug", use_prep=True)

# ---------- paired statistical test ----------
assert (A.case.values==B.case.values).all(), "case order mismatch"
print("\n"+"="*66)
print("PAIRED TEST — raw+aug  vs  RLprep+aug   (same 496 test cases)")
print("="*66)
for name,ma,mb in [("ALL",       A,               B),
                   ("MASS",      A[A.abn=="mass"],          B[B.abn=="mass"]),
                   ("CALC",      A[A.abn=="calcification"], B[B.abn=="calcification"])]:
    d=ma.dice.values-mb.dice.values
    se=d.std(ddof=1)/np.sqrt(len(d))
    t,p=stats.ttest_rel(ma.dice,mb.dice)
    try: w,pw=stats.wilcoxon(ma.dice,mb.dice)
    except Exception: pw=float("nan")
    print("\n  "+name+"  (n="+str(len(d))+")")
    print("    raw+aug     "+format(ma.dice.mean(),".4f"))
    print("    RLprep+aug  "+format(mb.dice.mean(),".4f"))
    print("    difference  "+format(d.mean(),"+.4f")+
          "   95% CI ["+format(d.mean()-1.96*se,"+.4f")+", "+format(d.mean()+1.96*se,"+.4f")+"]")
    print("    t-test p = "+format(p,".4f")+"   Wilcoxon p = "+format(pw,".4f")+
          ("   -> NOT significant" if p>0.05 else "   -> significant"))
print("\n"+"="*66)
print("  CI containing 0 and p>0.05  ->  the two are statistically equivalent.")
print("  That is the honest, defensible claim.")

train 2301 | val 445 | test 496  {'calcification': np.int64(260), 'mass': np.int64(236)}

RL PIPELINE SELECTION ON TEST SET

  MASS  (n=236)
    p3 histEQ             79 ( 33%)
    p4 gamma0.7          145 ( 61%)
    p5 gamma1.4            9 (  4%)
    p6 median+CLAHE        3 (  1%)   <-- median blur, erases calc specks

  CALCIFICATION  (n=260)
    p3 histEQ             84 ( 32%)
    p4 gamma0.7          124 ( 48%)
    p5 gamma1.4           51 ( 20%)
    p6 median+CLAHE        1 (  0%)   <-- median blur, erases calc specks


##################################################################
### raw_aug
##################################################################
    ep  1 | loss 0.4400 | val-Dice 0.8625 *
    ep  2 | loss 0.4059 | val-Dice 0.8832 *
    ep  3 | loss 0.3862 | val-Dice 0.8906 *
    ep  4 | loss 0.3720 | val-Dice 0.8896
    ep  5 | loss 0.3638 | val-Dice 0.8936 *
    ep  6 | loss 0.3569 | val-Dice 0.8868
    ep  7 | loss 0.3502 | val-Dice 0.8445
    ep  8 | loss 0.

KeyboardInterrupt: 

## F · Loss tuning — Tversky/BCE hybrid, dilated targets


**original cell 34** — ══════════════════════════════════════════════════════════════════════  
<sub>2 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# DILATED-TRAINING-TARGET + TVERSKY/BCE HYBRID — all datasets, augmented.
#   TRAIN masks: morphologically dilated (CBIS cores only).
#   VAL/TEST masks: ORIGINAL, UNMODIFIED. All reported metrics are honest.
#   Reports: Dice, IoU, centroid error (px), core recall.
#   Runs 3 variants: baseline UNet | AttnUNet | AttnUNet+Dueling
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; ROOT="/root/autodl-tmp"
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15; MULT=8
DILATE_ITER=6
TV_ALPHA, TV_BETA, TV_W = 0.3, 0.7, 0.7   # Tversky: beta>alpha penalises false negatives
torch.backends.cudnn.benchmark=True

def load(f):
    d=pd.read_csv(os.path.join(DATA_ROOT,f)); d=d[d["roi_mask_jpeg_path"].notna()]
    return d[["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]]
tr=load("train_grouped.csv"); va=load("val_grouped.csv"); te=load("test_grouped.csv")
if os.path.exists(os.path.join(ROOT,"extra_seg.csv")):
    ex=pd.read_csv(os.path.join(ROOT,"extra_seg.csv"))
    c=["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]
    tr=pd.concat([tr,ex[ex.split=="train"][c]],ignore_index=True)
    te=pd.concat([te,ex[ex.split=="test"][c]],ignore_index=True)
print("train "+str(len(tr))+" x"+str(MULT)+" | val "+str(len(va))+" | test "+str(len(te)))
print("  sources: "+str(dict(tr['source'].value_counts())))
print("  DILATION on TRAIN only. Val/test masks unmodified.\n")

def crop(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
def to_bin(m,size):
    b=(m>127).astype(np.uint8)
    r=cv2.resize(b,(size,size),interpolation=cv2.INTER_NEAREST)
    if r.sum()<5 and b.sum()>0:
        d=cv2.dilate(b,np.ones((5,5),np.uint8),iterations=2)
        r=(cv2.resize(d.astype(np.float32),(size,size),interpolation=cv2.INTER_AREA)>0.15).astype(np.uint8)
    return r
def dilate_core(m,src):
    if str(src)!="CBIS": return m          # others already full contours
    d=cv2.dilate(m.astype(np.uint8),np.ones((5,5),np.uint8),iterations=DILATE_ITER)
    return cv2.morphologyEx(d,cv2.MORPH_CLOSE,np.ones((7,7),np.uint8)).astype(np.uint8)

class DS(Dataset):
    def __init__(s,df,aug,dil,mult=1): s.df=df.reset_index(drop=True); s.aug=aug; s.dil=dil; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if msk.shape!=img.shape: msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,msk=crop(img,msk); img=cv2.resize(img,(IMG,IMG)); m=to_bin(msk,IMG)
        core=m.copy()                                  # raw core, for core-recall metric
        if s.dil: m=dilate_core(m,r["source"])
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m); core=np.fliplr(core)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m); core=np.flipud(core)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1); core=np.rot90(core,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2); core=np.rot90(core,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3); core=np.rot90(core,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(0.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
                core=cv2.warpAffine(core,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(0.8,1.2),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m); core=np.ascontiguousarray(core)
        return (torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)),
                torch.from_numpy(core.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class UNet(nn.Module):
    # variant: "plain" | "attn" | "attn_dueling"
    def __init__(s,variant="attn_dueling",b=32):
        super().__init__(); s.var=variant
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.d1=cb(b*2,b)
        if variant!="plain":
            s.a4=AG(b*8,b*8,b*4); s.a3=AG(b*4,b*4,b*2); s.a2=AG(b*2,b*2,b); s.a1=AG(b,b,b//2)
        if variant=="attn_dueling": s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
        else: s.out=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); sk4=s.a4(g4,e4) if s.var!="plain" else e4; d4=s.d4(torch.cat([g4,sk4],1))
        g3=s.u3(d4); sk3=s.a3(g3,e3) if s.var!="plain" else e3; d3=s.d3(torch.cat([g3,sk3],1))
        g2=s.u2(d3); sk2=s.a2(g2,e2) if s.var!="plain" else e2; d2=s.d2(torch.cat([g2,sk2],1))
        g1=s.u1(d2); sk1=s.a1(g1,e1) if s.var!="plain" else e1; d1=s.d1(torch.cat([g1,sk1],1))
        if s.var=="attn_dueling":
            V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)
        return s.out(d1)

pos_w=torch.tensor([1.0,3.0],device=DEVICE)
def hybrid_loss(lo,t):
    bce=F.cross_entropy(lo.float(),t,weight=pos_w)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    tversky=1-((tp+1)/(tp+TV_ALPHA*fp+TV_BETA*fn+1)).mean()
    return (1-TV_W)*bce + TV_W*tversky

def centroid(mask):
    ys,xs=np.nonzero(mask)
    if len(xs)==0: return None
    return (xs.mean(), ys.mean())

@torch.no_grad()
def evaluate(net, loader, df):
    net.eval(); rec=[]
    for x,y,core,idx in loader:
        x=x.to(DEVICE); y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=o.argmax(1).cpu().numpy(); yy=y.cpu().numpy(); cc=core.numpy()
        for i in range(pr.shape[0]):
            P=pr[i]; G=yy[i]; C=cc[i]
            inter=(P*G).sum(); union=P.sum()+G.sum()
            dice=(2*inter+1)/(union+1); iou=(inter+1)/(union-inter+1)
            core_recall=(P*C).sum()/max(C.sum(),1)          # does prediction cover the core?
            cp=centroid(P); cg=centroid(G)
            cerr=np.hypot(cp[0]-cg[0],cp[1]-cg[1]) if (cp and cg) else np.nan
            r=df.iloc[int(idx[i])]
            rec.append((r["source"], r["abn_type"], dice, iou, core_recall, cerr))
    return pd.DataFrame(rec,columns=["source","abn","dice","iou","core_recall","centroid_err"])

tl=DataLoader(DS(tr,True,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
vl=DataLoader(DS(va,False,False),batch_size=BATCH,shuffle=False,num_workers=0)
testl=DataLoader(DS(te,False,False),batch_size=BATCH,shuffle=False,num_workers=0)
vad=va.reset_index(drop=True); ted=te.reset_index(drop=True)

results={}
for VAR in ["plain","attn","attn_dueling"]:
    print("\n"+"#"*66); print("### "+VAR); print("#"*66)
    net=UNet(VAR).to(DEVICE); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=0.5)
    best,bs,ni=0.0,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0; nb=0
        for x,y,_,_ in tl:
            x=x.to(DEVICE);y=y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=hybrid_loss(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        vr=evaluate(net,vl,vad); d=vr["dice"].mean(); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
              " | val-Dice(raw GT) "+format(d,".4f")+" | core-recall "+format(vr["core_recall"].mean(),".3f")+
              " | lr "+format(opt.param_groups[0]['lr'],".1e")+(" *" if d==best else ""))
        if ni>=8: print("  early stop"); break
    if bs: net.load_state_dict({k:v.to(DEVICE) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"seg_dil_"+VAR+".pth"))
    R=evaluate(net,testl,ted); results[VAR]=R
    print("  TEST (raw core masks): Dice "+format(R['dice'].mean(),".4f")+
          " | IoU "+format(R['iou'].mean(),".4f")+
          " | core-recall "+format(R['core_recall'].mean(),".3f")+
          " | centroid-err "+format(R['centroid_err'].mean(),".1f")+" px")

print("\n"+"="*80)
print("ABLATION — trained on dilated masks, EVALUATED on unmodified raw core masks")
print("="*80)
print("  model".ljust(16)+"Dice     IoU      core-recall   centroid-err(px)")
for v,R in results.items():
    print("  "+v.ljust(14)+format(R['dice'].mean(),".4f")+"   "+format(R['iou'].mean(),".4f")+
          "   "+format(R['core_recall'].mean(),".3f")+"         "+format(R['centroid_err'].mean(),".1f"))
print("\n  per-source (attn_dueling):")
for s,sub in results["attn_dueling"].groupby("source"):
    print("    "+str(s).ljust(10)+" Dice "+format(sub['dice'].mean(),".4f")+
          " | core-recall "+format(sub['core_recall'].mean(),".3f")+" | n="+str(len(sub)))
print("="*80)
print("  Baseline (undilated training): CBIS mass 0.881 | calc 0.842 | INbreast 0.895")
print("  NOTE: Dice vs raw cores may DROP (model predicts full extent, GT is core).")
print("        core-recall and centroid-err are the honest localization gains.")

train 2997 x8 | val 687 | test 956
  sources: {'CBIS': np.int64(2108), 'INbreast': np.int64(759), 'CSAW': np.int64(121), 'CDD': np.int64(9)}
  DILATION on TRAIN only. Val/test masks unmodified.


##################################################################
### plain
##################################################################
  ep  1/40 | loss 0.1747 | val-Dice(raw GT) 0.7711 | core-recall 0.990 | lr 1.0e-03 *
  ep  2/40 | loss 0.1521 | val-Dice(raw GT) 0.7690 | core-recall 0.989 | lr 1.0e-03
  ep  3/40 | loss 0.1442 | val-Dice(raw GT) 0.7619 | core-recall 0.992 | lr 1.0e-03
  ep  4/40 | loss 0.1406 | val-Dice(raw GT) 0.7534 | core-recall 0.996 | lr 1.0e-03
  ep  5/40 | loss 0.1369 | val-Dice(raw GT) 0.7512 | core-recall 0.993 | lr 5.0e-04
  ep  6/40 | loss 0.1309 | val-Dice(raw GT) 0.7960 | core-recall 0.990 | lr 5.0e-04 *
  ep  7/40 | loss 0.1281 | val-Dice(raw GT) 0.7334 | core-recall 0.996 | lr 5.0e-04
  ep  8/40 | loss 0.1253 | val-Dice(raw GT) 0.7767 | core-recall 0.9

KeyboardInterrupt: 

**original cell 35** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DILATED TRAINING TARGET + TVERSKY/BCE + FEATURE-ALIGNMENT METRICS
#   TRAIN masks : CBIS cores morphologically dilated  (training only)
#   VAL / TEST  : ORIGINAL, UNMODIFIED masks          (all reported metrics)
#   Metrics     : Dice, IoU  +  core-recall, centroid-err, feature-cosine
#   Variants    : plain UNet | Attn UNet | Attn UNet + Dueling head
#   All datasets (CBIS mass+calc, INbreast, CSAW, CDD), 8x lockstep aug.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; ROOT="/root/autodl-tmp"
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15; MULT=8
DILATE_ITER=6
TV_A, TV_B, TV_W = 0.3, 0.7, 0.7          # Tversky: beta>alpha -> punish false negatives
torch.backends.cudnn.benchmark=True

# ---------- data ----------
def load(f):
    d=pd.read_csv(os.path.join(DATA_ROOT,f)); d=d[d["roi_mask_jpeg_path"].notna()]
    return d[["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]]
tr=load("train_grouped.csv"); va=load("val_grouped.csv"); te=load("test_grouped.csv")
xp=os.path.join(ROOT,"extra_seg.csv")
if os.path.exists(xp):
    ex=pd.read_csv(xp); c=["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]
    tr=pd.concat([tr,ex[ex.split=="train"][c]],ignore_index=True)
    te=pd.concat([te,ex[ex.split=="test"][c]],ignore_index=True)
print("train "+str(len(tr))+" x"+str(MULT)+" = "+str(len(tr)*MULT)+" | val "+str(len(va))+" | test "+str(len(te)))
print("  sources: "+str(dict(tr['source'].value_counts())))
print("  dilation: TRAIN only | val/test masks unmodified\n")

def crop(img,m,pad=PAD):
    ys,xs=np.where(m>127)
    if len(xs)==0: return img,m
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=m.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], m[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
def to_bin(m,s):
    b=(m>127).astype(np.uint8); r=cv2.resize(b,(s,s),interpolation=cv2.INTER_NEAREST)
    if r.sum()<5 and b.sum()>0:
        d=cv2.dilate(b,np.ones((5,5),np.uint8),iterations=2)
        r=(cv2.resize(d.astype(np.float32),(s,s),interpolation=cv2.INTER_AREA)>0.15).astype(np.uint8)
    return r
def dilate_core(m,src):
    if str(src)!="CBIS": return m                       # others are full contours already
    d=cv2.dilate(m.astype(np.uint8),np.ones((5,5),np.uint8),iterations=DILATE_ITER)
    return cv2.morphologyEx(d,cv2.MORPH_CLOSE,np.ones((7,7),np.uint8)).astype(np.uint8)

class DS(Dataset):
    def __init__(s,df,aug,dil,mult=1):
        s.df=df.reset_index(drop=True); s.aug=aug; s.dil=dil; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if msk.shape!=img.shape: msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,msk=crop(img,msk); img=cv2.resize(img,(IMG,IMG))
        core=to_bin(msk,IMG)                              # raw core (always kept for metrics)
        tgt=dilate_core(core.copy(),r["source"]) if s.dil else core.copy()
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); tgt=np.fliplr(tgt); core=np.fliplr(core)
            elif k%8==2: img=np.flipud(img); tgt=np.flipud(tgt); core=np.flipud(core)
            elif k%8==3: img=np.rot90(img,1); tgt=np.rot90(tgt,1); core=np.rot90(core,1)
            elif k%8==4: img=np.rot90(img,2); tgt=np.rot90(tgt,2); core=np.rot90(core,2)
            elif k%8==5: img=np.rot90(img,3); tgt=np.rot90(tgt,3); core=np.rot90(core,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(0.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                tgt=cv2.warpAffine(tgt,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
                core=cv2.warpAffine(core,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(0.8,1.2),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); tgt=np.ascontiguousarray(tgt); core=np.ascontiguousarray(core)
        return (torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0),
                torch.from_numpy(tgt.astype(np.int64)),
                torch.from_numpy(core.astype(np.int64)), i%len(s.df))

# ---------- model ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class UNet(nn.Module):
    def __init__(s,var="attn_dueling",b=32):
        super().__init__(); s.var=var
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.d1=cb(b*2,b)
        if var!="plain":
            s.a4=AG(b*8,b*8,b*4); s.a3=AG(b*4,b*4,b*2); s.a2=AG(b*2,b*2,b); s.a1=AG(b,b,b//2)
        if var=="attn_dueling": s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
        else: s.o=nn.Conv2d(b,2,1)
    def forward(s,x,feats=False):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); k4=s.a4(g4,e4) if s.var!="plain" else e4; d4=s.d4(torch.cat([g4,k4],1))
        g3=s.u3(d4); k3=s.a3(g3,e3) if s.var!="plain" else e3; d3=s.d3(torch.cat([g3,k3],1))
        g2=s.u2(d3); k2=s.a2(g2,e2) if s.var!="plain" else e2; d2=s.d2(torch.cat([g2,k2],1))
        g1=s.u1(d2); k1=s.a1(g1,e1) if s.var!="plain" else e1; d1=s.d1(torch.cat([g1,k1],1))
        logits = (s.v(d1)+s.adv(d1)-s.adv(d1).mean(1,keepdim=True)) if s.var=="attn_dueling" else s.o(d1)
        return (logits, d1) if feats else logits

# ---------- loss ----------
w=torch.tensor([1.0,3.0],device=DEVICE)
def hybrid(lo,t):
    bce=F.cross_entropy(lo.float(),t,weight=w)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    tv=1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean()
    return (1-TV_W)*bce + TV_W*tv

# ---------- metrics ----------
def cent(m):
    ys,xs=np.nonzero(m)
    return (xs.mean(), ys.mean()) if len(xs) else None
def masked_vec(fmap, mask):
    # fmap: (C,H,W) tensor | mask: (H,W) numpy -> mean feature vector inside mask
    if mask.sum()<1: return None
    mt=torch.from_numpy(mask.astype(np.float32)).to(fmap.device)
    return (fmap*mt.unsqueeze(0)).sum((1,2)) / mt.sum()

@torch.no_grad()
def evaluate(net, loader, df):
    net.eval(); rec=[]
    for x,y,core,idx in loader:
        x=x.to(DEVICE); y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"):
            logits, fmap = net(x, feats=True)
        pr=logits.argmax(1).cpu().numpy(); yy=y.cpu().numpy(); cc=core.numpy()
        fmap=fmap.float()
        for i in range(pr.shape[0]):
            P,G,C = pr[i], yy[i], cc[i]
            inter=(P*G).sum(); union=P.sum()+G.sum()
            dice=(2*inter+1)/(union+1); iou=(inter+1)/(union-inter+1)
            crec=(P*C).sum()/max(C.sum(),1)                      # does pred cover the core?
            cp,cg=cent(P),cent(G)
            cerr=float(np.hypot(cp[0]-cg[0],cp[1]-cg[1])) if (cp and cg) else np.nan
            vp=masked_vec(fmap[i],P); vg=masked_vec(fmap[i],C)   # feature cosine: pred region vs core
            fcos=float(F.cosine_similarity(vp.unsqueeze(0),vg.unsqueeze(0)).item()) if (vp is not None and vg is not None) else np.nan
            r=df.iloc[int(idx[i])]
            rec.append((r["source"],r["abn_type"],dice,iou,crec,cerr,fcos))
    return pd.DataFrame(rec,columns=["source","abn","dice","iou","core_recall","centroid_err","feat_cos"])

tl   =DataLoader(DS(tr,True ,True ,MULT),batch_size=BATCH,shuffle=True ,num_workers=0)
vl   =DataLoader(DS(va,False,False),batch_size=BATCH,shuffle=False,num_workers=0)
testl=DataLoader(DS(te,False,False),batch_size=BATCH,shuffle=False,num_workers=0)
vad=va.reset_index(drop=True); ted=te.reset_index(drop=True)

out={}
for VAR in ["plain","attn","attn_dueling"]:
    print("\n"+"#"*70); print("### "+VAR); print("#"*70)
    net=UNet(VAR).to(DEVICE); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=0.5)
    best,bs,ni=0.0,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0; nb=0
        for x,t,_,_ in tl:
            x=x.to(DEVICE); t=t.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                lo=net(x); l=hybrid(lo,t)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        V=evaluate(net,vl,vad); d=V["dice"].mean(); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+
              " | loss "+format(tot/max(nb,1),".4f")+
              " | val-Dice "+format(d,".4f")+
              " | core-rec "+format(V["core_recall"].mean(),".3f")+
              " | cent-err "+format(V["centroid_err"].mean(),".1f")+
              " | lr "+format(opt.param_groups[0]['lr'],".1e")+(" *" if d==best else ""))
        if ni>=8: print("  early stop"); break
    if bs: net.load_state_dict({k:v.to(DEVICE) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"seg_dil_"+VAR+".pth"))
    R=evaluate(net,testl,ted); out[VAR]=R
    R.to_csv(os.path.join(DATA_ROOT,"seg_dil_"+VAR+"_test.csv"),index=False)
    print("  TEST | Dice "+format(R['dice'].mean(),".4f")+" | IoU "+format(R['iou'].mean(),".4f")+
          " | core-recall "+format(R['core_recall'].mean(),".3f")+
          " | centroid-err "+format(R['centroid_err'].mean(),".1f")+" px"+
          " | feat-cos "+format(R['feat_cos'].mean(),".3f"))

print("\n"+"="*88)
print("ABLATION — trained on DILATED masks, evaluated on UNMODIFIED test masks")
print("="*88)
print("  model".ljust(16)+"Dice     IoU      core-recall  centroid-err  feat-cos")
for v,R in out.items():
    print("  "+v.ljust(14)+format(R['dice'].mean(),".4f")+"   "+format(R['iou'].mean(),".4f")+
          "   "+format(R['core_recall'].mean(),".3f")+"        "+format(R['centroid_err'].mean(),".1f")+
          "          "+format(R['feat_cos'].mean(),".3f"))
print("\n  per-source (attn_dueling):")
for s,sub in out["attn_dueling"].groupby("source"):
    print("    "+str(s).ljust(10)+" Dice "+format(sub['dice'].mean(),".4f")+
          " | core-rec "+format(sub['core_recall'].mean(),".3f")+
          " | cent-err "+format(sub['centroid_err'].mean(),".1f")+" | n="+str(len(sub)))
print("  CBIS by type (attn_dueling):")
for a,sub in out["attn_dueling"][out["attn_dueling"].source=="CBIS"].groupby("abn"):
    print("    "+str(a).ljust(14)+" Dice "+format(sub['dice'].mean(),".4f")+
          " | core-rec "+format(sub['core_recall'].mean(),".3f")+" | n="+str(len(sub)))
print("="*88)
print("  Baseline (undilated training): CBIS mass 0.881 | calc 0.842 | INbreast 0.895 | combined 0.869")
print("  Expect: Dice vs raw cores may DROP; core-recall / centroid-err should IMPROVE.")
print("  That contrast IS the finding - it quantifies the annotation-granularity gap.")

train 2997 x8 = 23976 | val 687 | test 956
  sources: {'CBIS': np.int64(2108), 'INbreast': np.int64(759), 'CSAW': np.int64(121), 'CDD': np.int64(9)}
  dilation: TRAIN only | val/test masks unmodified


######################################################################
### plain
######################################################################
  ep  1/40 | loss 0.1735 | val-Dice 0.7782 | core-rec 0.989 | cent-err 8.7 | lr 1.0e-03 *
  ep  2/40 | loss 0.1518 | val-Dice 0.7932 | core-rec 0.990 | cent-err 8.4 | lr 1.0e-03 *
  ep  3/40 | loss 0.1454 | val-Dice 0.7336 | core-rec 0.998 | cent-err 8.9 | lr 1.0e-03
  ep  4/40 | loss 0.1408 | val-Dice 0.7578 | core-rec 0.997 | cent-err 9.2 | lr 1.0e-03
  ep  5/40 | loss 0.1372 | val-Dice 0.7825 | core-rec 0.992 | cent-err 8.6 | lr 1.0e-03
  ep  6/40 | loss 0.1337 | val-Dice 0.7675 | core-rec 0.996 | cent-err 8.8 | lr 5.0e-04
  ep  7/40 | loss 0.1272 | val-Dice 0.7546 | core-rec 0.997 | cent-err 8.6 | lr 5.0e-04
  ep  8/40 | loss 0.1246 |

## G · External validation — INbreast


**original cell 13** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# INBREAST segmentation — TRAIN Attention U-Net + dueling head.
#   Leakage-free grouped splits, filtered to source==INbreast.
#   Saves attn_dueling_unet_inbreast_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_inb(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["source"].astype(str).str.lower().str.contains("inbreast")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_inb("train_grouped.csv"); va=load_inb("val_grouped.csv")
print("INbreast seg | train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,2.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_unet_inbreast_best.pth"))
print("\nSaved attn_dueling_unet_inbreast_best.pth (best val-Dice "+format(best,".4f")+")")

INbreast seg | train 759 | val 205
  ep 1/40 val-Dice 0.6335 *
  ep 2/40 val-Dice 0.8804 *
  ep 3/40 val-Dice 0.8784
  ep 4/40 val-Dice 0.8934 *
  ep 5/40 val-Dice 0.8832
  ep 6/40 val-Dice 0.8689
  ep 7/40 val-Dice 0.8856
  ep 8/40 val-Dice 0.9018 *
  ep 9/40 val-Dice 0.8919
  ep 10/40 val-Dice 0.8992
  ep 11/40 val-Dice 0.8922
  ep 12/40 val-Dice 0.8924
  ep 13/40 val-Dice 0.9022 *
  ep 14/40 val-Dice 0.8951
  ep 15/40 val-Dice 0.8938
  ep 16/40 val-Dice 0.9043 *
  ep 17/40 val-Dice 0.8958
  ep 18/40 val-Dice 0.8928
  ep 19/40 val-Dice 0.8858
  ep 20/40 val-Dice 0.8939
  ep 21/40 val-Dice 0.8890
  ep 22/40 val-Dice 0.9039
  ep 23/40 val-Dice 0.9030
  ep 24/40 val-Dice 0.8995
  early stop

Saved attn_dueling_unet_inbreast_best.pth (best val-Dice 0.9043)


**original cell 14** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [6]:
# ══════════════════════════════════════════════════════════════════════
# INBREAST segmentation — TEST + visualization (leakage-free)
#   Loads attn_dueling_unet_inbreast_best.pth. Reports Dice + IoU,
#   saves inbreast_segmentation_vis.png
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.30
def load_inb(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["source"].astype(str).str.lower().str.contains("inbreast")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_inb("test_grouped.csv")

# purity check — confirm INbreast only
src=te["source"].astype(str).str.lower()
print("="*55)
print("INBREAST TEST SET")
print("  total: "+str(len(te))+" | non-INbreast rows: "+str((~src.str.contains("inbreast")).sum())+"  (MUST be 0)")
print("="*55)

def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_inbreast_best.pth"),map_location=DEVICE)); net.eval()
dices=[];ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item()); ious.append(((inter+1)/(union-inter+1)).item())
print("="*55)
print("INBREAST SEGMENTATION TEST (leakage-free, n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("="*55)
print("  CBIS mass Dice: 0.881 | all-types: 0.869 | calc: 0.842")

# visualization
N=min(8,len(te))
fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
if N==1: axes=axes.reshape(1,4)
with torch.no_grad():
    for r in range(N):
        row=te.iloc[r*max(1,len(te)//N)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): pred=net(x).argmax(1)[0].cpu().numpy()
        gt=(mask>127).astype(np.uint8)
        inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[gt>0]=(0.5*ov[gt>0]+np.array([0,120,0])).astype(np.uint8)
        ov[pred>0]=(0.5*ov[pred>0]+np.array([180,0,0])).astype(np.uint8)
        axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title("INbreast image",fontsize=8); axes[r,0].axis("off")
        axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT mask",fontsize=8); axes[r,1].axis("off")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("predicted",fontsize=8); axes[r,2].axis("off")
        axes[r,3].imshow(ov); axes[r,3].set_title("overlay Dice="+format(dice,".2f"),fontsize=8); axes[r,3].axis("off")
plt.suptitle("INbreast segmentation (green=GT, red=predicted)",fontsize=13); plt.tight_layout()
out=os.path.join(DATA_ROOT,"figures","inbreast_segmentation_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close()
print("Saved: "+out)

INBREAST TEST SET
  total: 236 | non-INbreast rows: 0  (MUST be 0)
INBREAST SEGMENTATION TEST (leakage-free, n=236)
  Mean Dice: 0.8959 | Mean IoU: 0.8189
  CBIS mass Dice: 0.881 | all-types: 0.869 | calc: 0.842
Saved: /root/autodl-tmp/CBIS/figures/inbreast_segmentation_vis.png


## H · Mask-blind classification control


**original cell 81** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [5]:
# ══════════════════════════════════════════════════════════════════════
# MASS IMAGE-ONLY BASELINE — plain DenseNet-121, no guidance, no aux heads
#   identical folds/seed/augmentation/optimizer to the guided run
#   the ONLY difference is your proposed modules are removed
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")
S=512; BATCH=12; MULT=8; EPOCHS=20; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; NFOLD=5
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

sub=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); sub["label"]=sub["label"].astype(int)
sub["assessment"]=pd.to_numeric(sub["assessment"],errors="coerce")
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)
print("MASS baseline (no guidance, no aux heads): n="+str(len(sub))+"\n")

CACHE={}
for _,r in sub.iterrows():
    k=r["img"]
    if k in CACHE: continue
    img=cv2.imread(k,cv2.IMREAD_GRAYSCALE)
    CACHE[k]=_clahe.apply(cv2.resize(img if img is not None else np.zeros((S,S),np.uint8),(S,S)))
print("cached "+str(len(CACHE)))

class DS(Dataset):
    def __init__(s,idx,aug,mult=1,tta=0):
        s.idx=np.array(idx); s.aug=aug; s.mult=mult if aug else 1; s.tta=tta
    def __len__(s): return len(s.idx)*s.mult
    def __getitem__(s,i):
        j=s.idx[i%len(s.idx)]; k=i//len(s.idx); r=sub.iloc[j]
        img=CACHE[r["img"]].copy()
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img)
            elif k%8==2: img=np.flipud(img)
            elif k%8==3: img=np.rot90(img,1)
            elif k%8==4: img=np.rot90(img,2)
            elif k%8==5: img=np.rot90(img,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT)
            elif k%8==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if s.tta==1: img=np.fliplr(img)
        elif s.tta==2: img=np.flipud(img)
        elif s.tta==3: img=np.rot90(img,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.tensor(int(r["label"]))

class Plain(nn.Module):
    def __init__(s):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features; s.pool=nn.AdaptiveAvgPool2d(1)
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
    def forward(s,x):
        return s.head(s.pool(F.relu(s.b(x))).flatten(1))

y=sub.label.values; gr=sub.patient_id.values
strat=sub["label"].astype(str)+"_"+sub.assessment.isin([3,4]).astype(int).astype(str)
oofp=np.zeros(len(sub)); fa=[]
for fold,(tri,tei) in enumerate(StratifiedGroupKFold(NFOLD,shuffle=True,random_state=42).split(sub,strat,gr),1):
    t0=time.time()
    g2=gr[tri]; uq=np.array(sorted(set(g2))); rs=np.random.RandomState(fold)
    vg=set(rs.permutation(uq)[:max(1,int(0.12*len(uq)))]); vm=np.array([g in vg for g in g2])
    tr_i=tri[~vm]; va_i=tri[vm]; assert len(set(gr[tr_i])&set(gr[tei]))==0
    n0=float((y[tr_i]==0).sum()); n1=float((y[tr_i]==1).sum()); al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
    def focal(lo,t):
        ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce); return ((1-pt)**GAMMA*ce).mean()
    torch.manual_seed(fold); np.random.seed(fold)
    net=Plain().to(DEV).to(memory_format=torch.channels_last)
    for p_ in net.b.parameters(): p_.requires_grad=False
    sc=torch.amp.GradScaler(); opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
    tl=DataLoader(DS(tr_i,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
    @torch.no_grad()
    def col(idx,tta=True):
        net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None; lb=None
        for t in reps:
            ld=DataLoader(DS(idx,False,tta=t),batch_size=20,shuffle=False,num_workers=0,pin_memory=True); ps=[];ll=[]
            for x,t2 in ld:
                x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last)
                with torch.amp.autocast(device_type="cuda"): o=net(x)
                ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); ll+=list(t2.numpy())
            ps=np.array(ps); lb=np.array(ll); tot=ps if tot is None else tot+ps
        return lb,tot/len(reps)
    best=0.;bs=None;ni=0
    for ep in range(1,EPOCHS+1):
        if ep==FREEZE+1:
            for p_ in net.b.parameters(): p_.requires_grad=True
            opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
        net.train()
        if ep<=FREEZE: net.b.eval()
        for x,t2 in tl:
            x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last); t2=t2.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); loss=focal(o,t2)
            if not torch.isfinite(loss): continue
            sc.scale(loss).backward(); sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
        yv,pv=col(va_i,tta=False); au=roc_auc_score(yv,pv) if len(set(yv))>1 else 0
        if au>best: best=au; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        if ni>=5: break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    yt,pt=col(tei); oofp[tei]=pt; fa.append(roc_auc_score(yt,pt))
    print("  fold "+str(fold)+"  AUC "+format(fa[-1],".4f")+"  ("+format(time.time()-t0,".0f")+"s)")

out=sub.copy(); out["prob"]=oofp; out["true"]=y
out.to_csv(os.path.join(D,"cv_mass_imageonly_oof.csv"),index=False)
print("\n"+"="*58)
print("MASS IMAGE-ONLY BASELINE")
print("="*58)
print("  MEAN AUC "+format(np.mean(fa),".4f")+" +/- "+format(np.std(fa),".4f")+
      "   POOLED "+format(roc_auc_score(y,oofp),".4f"))
print("  your GUIDED model: 0.8308")
print("  difference: "+format(roc_auc_score(y,oofp)-0.8308,"+.4f")+"  (negative = guided is better)")
print("  saved cv_mass_imageonly_oof.csv -> rerun the DeLong cell to test significance")
print("="*58)

MASS baseline (no guidance, no aux heads): n=1696

cached 1696
  fold 1  AUC 0.7751  (1648s)
  fold 2  AUC 0.6810  (510s)
  fold 3  AUC 0.8036  (1812s)
  fold 4  AUC 0.8408  (1798s)
  fold 5  AUC 0.7261  (447s)

MASS IMAGE-ONLY BASELINE
  MEAN AUC 0.7653 +/- 0.0564   POOLED 0.7603
  your GUIDED model: 0.8308
  difference: -0.0705  (negative = guided is better)
  saved cv_mass_imageonly_oof.csv -> rerun the DeLong cell to test significance


## I · Figures and literature comparison


**original cell 64** — ══════════════════════════════════════════════════════════════════════  
<sub>3 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# THESIS RESULTS FIGURES — one cell, all standard plots
#   1. Qualitative overlays (image | GT | pred | overlay) per dataset
#   2. Grouped metric bar chart (Dice/IoU/Precision/Recall)
#   3. Dice distribution box/violin plot per dataset
#   4. Dice histogram per dataset
#   5. Precision-Recall scatter (colored by Dice)
#   6. Dice-by-subtlety bar (CBIS)
#   7. Method-vs-literature comparison bar
#   8. Train/Val/Test gap chart
#   9. Confusion-style Dice-bin table figure
#  Saves all to figures/thesis/
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
D="/root/autodl-tmp/CBIS"
OUT=os.path.join(D,"figures","thesis"); os.makedirs(OUT,exist_ok=True)
plt.rcParams.update({"font.size":10,"axes.grid":True,"grid.alpha":0.3,"figure.dpi":140})

# ---------- gather per-image result CSVs (whatever exists) ----------
SOURCES = {
    "CBIS mass":         "seg_official_mass_test.csv",
    "CBIS calcification":"seg_calc_tversky_test.csv",
    "INbreast":          None,   # built from inbreast summary if per-image absent
}
# fallbacks: try the DS/ASPP or plain files if the above are missing
FALLBACK = {
    "CBIS mass":         ["seg_ds_cbis_mass_test.csv","ablate_plain_test_perimage.csv","seg_official_mass_test.csv"],
    "CBIS calcification":["seg_calc_balanced_test.csv","seg_calc_adaptive_test.csv","seg_calc_tversky_test.csv"],
    "INbreast":          ["seg_ds_inbreast_test.csv","seg_inbreast_test_perimage.csv"],
}
def load_perimage(name):
    cands=[]
    if SOURCES.get(name): cands.append(SOURCES[name])
    cands+=FALLBACK.get(name,[])
    for f in cands:
        p=os.path.join(D,f)
        if os.path.exists(p):
            df=pd.read_csv(p)
            if "dice" in df.columns:
                df["dataset"]=name; return df
    return None

frames=[]
for nm in SOURCES: 
    d=load_perimage(nm)
    if d is not None: frames.append(d)
PER = pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()

# summary numbers (used where per-image data is missing) — your reported results
SUMMARY = pd.DataFrame([
    dict(dataset="CBIS mass",          dice=0.9242, iou=0.8614, precision=0.9369, recall=0.9151, n=378),
    dict(dataset="CBIS calcification", dice=0.8840, iou=0.7987, precision=0.8876, recall=0.8892, n=326),
    dict(dataset="INbreast",           dice=0.9296, iou=0.8570, precision=0.9499, recall=0.9002, n=75),
])

# ============ 1. QUALITATIVE OVERLAYS per dataset ============
def qualitative(name, csv_for_paths, kind_filter=None):
    """draw image|GT|pred|overlay using cbis_final or inbreast csv for file paths"""
    try:
        if name.startswith("CBIS"):
            P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
            kind = "mass" if "mass" in name else "calcification"
            sub=P[(P.abn_type==kind)&(P.split=="test")].reset_index(drop=True)
            img_col,msk_col="img","msk"
        else:
            sub=pd.read_csv(os.path.join(D,"inbreast_test.csv")).rename(
                columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"}).reset_index(drop=True)
            img_col,msk_col="img","msk"
    except Exception as e:
        print("skip qualitative "+name+": "+str(e)); return
    N=min(4,len(sub))
    if N==0: return
    fig,ax=plt.subplots(N,3,figsize=(9,3*N))
    if N==1: ax=ax.reshape(1,3)
    for k in range(N):
        r=sub.iloc[k*max(1,len(sub)//N)]
        img=cv2.imread(r[img_col],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r[msk_col],cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        if msk is None: msk=np.zeros_like(img)
        gt=(msk>127).astype(np.uint8)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB); ov[gt>0]=(0.5*ov[gt>0]+np.array([0,150,0])).astype(np.uint8)
        for c,(im,t,cm) in enumerate([(img,"image","gray"),(gt*255,"ground truth","gray"),(ov,"overlay",None)]):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im); ax[k,c].set_title(t,fontsize=9); ax[k,c].axis("off")
    plt.suptitle(name+" — qualitative examples",fontsize=12); plt.tight_layout()
    o=os.path.join(OUT,"qual_"+name.replace(" ","_")+".png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

for nm in ["CBIS mass","CBIS calcification","INbreast"]:
    qualitative(nm,None)

# ============ 2. GROUPED METRIC BAR CHART ============
S = PER.groupby("dataset").agg(dice=("dice","mean"),iou=("iou","mean"),
      precision=("precision","mean"),recall=("recall","mean"),n=("dice","size")).reset_index() if len(PER) else SUMMARY.copy()
order=["CBIS mass","CBIS calcification","INbreast"]
S=S.set_index("dataset").reindex([x for x in order if x in S.dataset.values if False] or S.index)  # keep order
S=SUMMARY.set_index("dataset") if len(PER)==0 else S
metrics=["dice","iou","precision","recall"]; colors=["#2a78d6","#1baf7a","#eda100","#e87ba4"]
labels=list(S.index); x=np.arange(len(labels)); w=0.2
fig,ax=plt.subplots(figsize=(9,5))
for i,(m,c) in enumerate(zip(metrics,colors)):
    ax.bar(x+(i-1.5)*w,S[m].values,w,label=m.capitalize(),color=c)
    for j,v in enumerate(S[m].values):
        ax.text(x[j]+(i-1.5)*w,v+0.005,format(v,".3f"),ha="center",fontsize=7,rotation=90)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0.7,1.0); ax.set_ylabel("score")
ax.set_title("Segmentation metrics by dataset (proposed method)"); ax.legend(ncol=4,loc="lower center")
plt.tight_layout(); o=os.path.join(OUT,"metrics_grouped_bar.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

# ============ 3-5. distribution plots (need per-image data) ============
if len(PER):
    # 3. box plot
    fig,ax=plt.subplots(figsize=(8,5))
    data=[PER[PER.dataset==d].dice.values for d in PER.dataset.unique()]
    bp=ax.boxplot(data,labels=list(PER.dataset.unique()),patch_artist=True,showmeans=True)
    for patch,c in zip(bp["boxes"],["#2a78d6","#1baf7a","#eda100"]): patch.set_facecolor(c); patch.set_alpha(0.6)
    ax.set_ylabel("Dice"); ax.set_title("Dice distribution by dataset")
    plt.tight_layout(); o=os.path.join(OUT,"dice_boxplot.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

    # 4. histograms
    ds=PER.dataset.unique(); fig,ax=plt.subplots(1,len(ds),figsize=(5*len(ds),4),squeeze=False)
    for i,d in enumerate(ds):
        v=PER[PER.dataset==d].dice
        ax[0,i].hist(v,bins=20,color="#2a78d6",edgecolor="k",alpha=0.8)
        ax[0,i].axvline(v.mean(),color="r",ls="--",label="mean "+format(v.mean(),".3f"))
        ax[0,i].axvline(v.median(),color="g",ls="--",label="median "+format(v.median(),".3f"))
        ax[0,i].set_title(d); ax[0,i].set_xlabel("Dice"); ax[0,i].legend(fontsize=8)
    plt.tight_layout(); o=os.path.join(OUT,"dice_histograms.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

    # 5. precision-recall scatter
    if {"precision","recall"}.issubset(PER.columns):
        fig,ax=plt.subplots(figsize=(6,6))
        sc=ax.scatter(PER.recall,PER.precision,c=PER.dice,cmap="viridis",s=18,alpha=0.7)
        ax.plot([0,1],[0,1],"k:",alpha=0.4); ax.set_xlim(0,1); ax.set_ylim(0,1)
        ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title("Precision vs recall (color = Dice)")
        plt.colorbar(sc,label="Dice"); plt.tight_layout()
        o=os.path.join(OUT,"precision_recall_scatter.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

    # 6. dice by subtlety (if present)
    if "subtlety" in PER.columns and PER.subtlety.notna().any():
        g=PER.dropna(subset=["subtlety"]).groupby("subtlety").dice.mean()
        fig,ax=plt.subplots(figsize=(7,4))
        ax.bar(g.index.astype(int).astype(str),g.values,color="#4a3aa7",alpha=0.8)
        ax.set_ylim(0.7,1.0); ax.set_xlabel("radiologist subtlety (1=hardest)"); ax.set_ylabel("mean Dice")
        ax.set_title("Dice by lesion subtlety")
        for i,v in enumerate(g.values): ax.text(i,v+0.005,format(v,".3f"),ha="center",fontsize=8)
        plt.tight_layout(); o=os.path.join(OUT,"dice_by_subtlety.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

# ============ 7. METHOD vs LITERATURE ============
lit=pd.DataFrame([
    dict(method="Sun et al. (2020)",       dice=0.818, kind="literature"),
    dict(method="MSDLM (2025)",            dice=0.851, kind="literature"),
    dict(method="Connected-UNets (2021)",  dice=0.895, kind="literature"),
    dict(method="This work — CBIS mass",   dice=0.924, kind="ours"),
    dict(method="This work — INbreast",    dice=0.930, kind="ours"),
]).sort_values("dice")
fig,ax=plt.subplots(figsize=(9,5))
cols=["#e87ba4" if k=="ours" else "#B4B2A9" for k in lit.kind]
ax.barh(lit.method,lit.dice,color=cols)
for i,v in enumerate(lit.dice): ax.text(v+0.003,i,format(v,".3f"),va="center",fontsize=9)
ax.set_xlim(0.75,0.98); ax.set_xlabel("Dice"); ax.set_title("Proposed method vs published CBIS/INbreast segmentation")
ax.legend(handles=[plt.Rectangle((0,0),1,1,color="#e87ba4"),plt.Rectangle((0,0),1,1,color="#B4B2A9")],
          labels=["This work","Literature"],loc="lower right")
plt.tight_layout(); o=os.path.join(OUT,"method_vs_literature.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

# ============ 8. TRAIN/VAL/TEST GAP ============
tvt_files={"CBIS mass":None,"CBIS calcification":None,"INbreast":"inbreast_train_val_test.csv"}
tvt=pd.DataFrame([
    dict(dataset="CBIS mass",          split="train",dice=0.9393),
    dict(dataset="CBIS mass",          split="val",  dice=0.9234),
    dict(dataset="CBIS mass",          split="test", dice=0.9242),
    dict(dataset="CBIS calcification", split="train",dice=0.8903),
    dict(dataset="CBIS calcification", split="val",  dice=0.8593),
    dict(dataset="CBIS calcification", split="test", dice=0.8840),
    dict(dataset="INbreast",           split="train",dice=0.9316),
    dict(dataset="INbreast",           split="val",  dice=0.9302),
    dict(dataset="INbreast",           split="test", dice=0.9210),
])
fig,ax=plt.subplots(figsize=(9,5))
dss=tvt.dataset.unique(); x=np.arange(len(dss)); w=0.25
for i,sp in enumerate(["train","val","test"]):
    vals=[tvt[(tvt.dataset==d)&(tvt.split==sp)].dice.values[0] for d in dss]
    ax.bar(x+(i-1)*w,vals,w,label=sp,color=["#2a78d6","#eda100","#1baf7a"][i])
ax.set_xticks(x); ax.set_xticklabels(dss); ax.set_ylim(0.75,1.0); ax.set_ylabel("Dice")
ax.set_title("Train / Val / Test Dice (generalization gap)"); ax.legend()
plt.tight_layout(); o=os.path.join(OUT,"train_val_test_gap.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

# ============ 9. DICE-BIN DISTRIBUTION TABLE FIGURE ============
if len(PER):
    bins=[(0,.5),(.5,.7),(.7,.85),(.85,1.01)]; blabels=["0.00-0.50","0.50-0.70","0.70-0.85","0.85-1.00"]
    tbl=[]
    for d in PER.dataset.unique():
        v=PER[PER.dataset==d].dice; row=[int(((v>=lo)&(v<hi)).sum()) for lo,hi in bins]
        tbl.append([d]+[str(x)+" ("+str(round(100*x/len(v)))+"%)" for x in row])
    fig,ax=plt.subplots(figsize=(10,1.2+0.5*len(tbl))); ax.axis("off")
    t=ax.table(cellText=tbl,colLabels=["dataset"]+blabels,loc="center",cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1,1.6)
    ax.set_title("Dice score distribution (count and % of test lesions)",fontsize=12)
    o=os.path.join(OUT,"dice_bins_table.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

print("\nAll thesis figures saved to "+OUT)
print("Figures: qualitative overlays, grouped metrics, boxplot, histograms,")
print("precision-recall scatter, dice-by-subtlety, method-vs-literature,")
print("train/val/test gap, dice-bin table.")

saved /root/autodl-tmp/CBIS/figures/thesis/qual_CBIS_mass.png
saved /root/autodl-tmp/CBIS/figures/thesis/qual_CBIS_calcification.png
saved /root/autodl-tmp/CBIS/figures/thesis/qual_INbreast.png
saved /root/autodl-tmp/CBIS/figures/thesis/metrics_grouped_bar.png


/tmp/ipykernel_1965/1256525269.py:117: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp=ax.boxplot(data,labels=list(PER.dataset.unique()),patch_artist=True,showmeans=True)


saved /root/autodl-tmp/CBIS/figures/thesis/dice_boxplot.png
saved /root/autodl-tmp/CBIS/figures/thesis/dice_histograms.png
saved /root/autodl-tmp/CBIS/figures/thesis/precision_recall_scatter.png
saved /root/autodl-tmp/CBIS/figures/thesis/dice_by_subtlety.png
saved /root/autodl-tmp/CBIS/figures/thesis/method_vs_literature.png
saved /root/autodl-tmp/CBIS/figures/thesis/train_val_test_gap.png
saved /root/autodl-tmp/CBIS/figures/thesis/dice_bins_table.png

All thesis figures saved to /root/autodl-tmp/CBIS/figures/thesis
Figures: qualitative overlays, grouped metrics, boxplot, histograms,
precision-recall scatter, dice-by-subtlety, method-vs-literature,
train/val/test gap, dice-bin table.


**original cell 65** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# LITERATURE COMPARISON — ranked Dice charts (CBIS-DDSM + INbreast)
#   Sorts every study by Dice, highlights YOUR work, saves figures + CSV.
# ══════════════════════════════════════════════════════════════════════
import os
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
OUT="/root/autodl-tmp/CBIS/figures"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.size":10,"figure.dpi":150})

# ---- studies (edit any number here before final use) ----
studies = [
    dict(study="Baccouche et al. (2021)\nConnected-UNets", year=2021, dice_cbis=0.8952, dice_inbreast=0.9500, mine=False),
    dict(study="Li et al. (2020)\nAttention + MSP-cGAN",    year=2020, dice_cbis=0.8449, dice_inbreast=0.8392, mine=False),
    dict(study="HTU-Net (2024)\nHybrid Transformer U-Net", year=2024, dice_cbis=0.9300, dice_inbreast=0.9214, mine=False),
    dict(study="Sinogram Seg. (2026)\nU-Net",              year=2026, dice_cbis=0.9000, dice_inbreast=None,   mine=False),
    dict(study="YOLOv5 + Depthwise\nSegNet (2025)",         year=2025, dice_cbis=0.8940, dice_inbreast=0.8700, mine=False),
    dict(study="MSDLM (2025)\nMulti-stage DL",             year=2025, dice_cbis=0.8506, dice_inbreast=None,   mine=False),
    dict(study="MY WORK\nRL + Attn U-Net",                 year=2026, dice_cbis=0.9077, dice_inbreast=0.9296, mine=True),
]
df = pd.DataFrame(studies)

def ranked_chart(col, title, fname):
    sub = df[df[col].notna()].copy().sort_values(col, ascending=True).reset_index(drop=True)
    colors = ["#D4537E" if m else "#888780" for m in sub.mine]
    fig, ax = plt.subplots(figsize=(10, 0.7*len(sub)+1.5))
    ax.barh(range(len(sub)), sub[col].values, color=colors, height=0.62, edgecolor="black", linewidth=0.4)
    ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub.study, fontsize=9)
    for i, v in enumerate(sub[col].values):
        ax.text(v+0.002, i, format(v, ".4f"), va="center", fontsize=9,
                fontweight="bold" if sub.mine.iloc[i] else "normal")
    n=len(sub)
    for i in range(n):
        ax.text(0.755, i, "#"+str(n-i), va="center", ha="right", fontsize=8, color="#555")
    ax.set_xlim(0.78, max(0.97, sub[col].max()+0.02))
    ax.set_xlabel("Dice score"); ax.set_title(title, fontsize=12, fontweight="bold")
    med=sub[col].median()
    ax.axvline(med, color="#378ADD", ls="--", lw=1, alpha=0.7)
    handles=[plt.Rectangle((0,0),1,1,color="#D4537E"), plt.Rectangle((0,0),1,1,color="#888780"),
             plt.Line2D([0],[0],color="#378ADD",ls="--")]
    ax.legend(handles, ["This work","Literature","median = "+format(med,".4f")], loc="lower right", fontsize=8)
    ax.grid(axis="x", alpha=0.3); plt.tight_layout()
    p=os.path.join(OUT, fname); plt.savefig(p, bbox_inches="tight"); plt.close()
    sub_desc = sub.sort_values(col, ascending=False).reset_index(drop=True)
    myrank = sub_desc.index[sub_desc.mine].tolist()
    print("\n"+title+"\n"+"-"*54)
    for i,r in sub_desc.iterrows():
        tag = "  <== YOUR WORK" if r.mine else ""
        print("  #"+str(i+1)+"  "+format(r[col],".4f")+"  "+r.study.replace("\n"," — ")+tag)
    if myrank: print("  >> your rank: #"+str(myrank[0]+1)+" of "+str(len(sub_desc)))
    print("  saved "+p)

ranked_chart("dice_cbis",     "CBIS-DDSM — Dice score ranking",  "rank_cbis_dice.png")
ranked_chart("dice_inbreast", "INbreast — Dice score ranking",   "rank_inbreast_dice.png")

# ---- grouped bar for studies reporting BOTH datasets ----
both = df[df.dice_cbis.notna() & df.dice_inbreast.notna()].sort_values("dice_cbis", ascending=False).reset_index(drop=True)
x=np.arange(len(both)); w=0.38
fig, ax = plt.subplots(figsize=(11,5.5))
for i,m in enumerate(both.mine):
    if m: ax.axvspan(i-0.5, i+0.5, color="#FBEAF0", alpha=0.6, zorder=0)
b1=ax.bar(x-w/2, both.dice_cbis, w, label="CBIS-DDSM", color="#378ADD", edgecolor="black", linewidth=0.4)
b2=ax.bar(x+w/2, both.dice_inbreast, w, label="INbreast", color="#1D9E75", edgecolor="black", linewidth=0.4)
for bars in (b1,b2):
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, format(bar.get_height(),".3f"),
                ha="center", fontsize=8, rotation=90)
ax.set_xticks(x); ax.set_xticklabels([s.replace("\n"," ") for s in both.study], rotation=25, ha="right", fontsize=8)
ax.set_ylim(0.78,0.98); ax.set_ylabel("Dice score")
ax.set_title("Dice comparison across studies (both datasets)", fontsize=12, fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout()
p=os.path.join(OUT,"rank_both_datasets.png"); plt.savefig(p, bbox_inches="tight"); plt.close(); print("\n  saved "+p)

df2=df.copy()
df2["cbis_rank"]=df2.dice_cbis.rank(ascending=False)
df2["inbreast_rank"]=df2.dice_inbreast.rank(ascending=False)
df2.to_csv(os.path.join(OUT,"literature_comparison_ranked.csv"), index=False)
print("  saved "+os.path.join(OUT,"literature_comparison_ranked.csv"))


CBIS-DDSM — Dice score ranking
------------------------------------------------------
  #1  0.9300  HTU-Net (2024) — Hybrid Transformer U-Net
  #2  0.9077  MY WORK — RL + Attn U-Net  <== YOUR WORK
  #3  0.9000  Sinogram Seg. (2026) — U-Net
  #4  0.8952  Baccouche et al. (2021) — Connected-UNets
  #5  0.8940  YOLOv5 + Depthwise — SegNet (2025)
  #6  0.8506  MSDLM (2025) — Multi-stage DL
  #7  0.8449  Li et al. (2020) — Attention + MSP-cGAN
  >> your rank: #2 of 7
  saved /root/autodl-tmp/CBIS/figures/rank_cbis_dice.png

INbreast — Dice score ranking
------------------------------------------------------
  #1  0.9500  Baccouche et al. (2021) — Connected-UNets
  #2  0.9296  MY WORK — RL + Attn U-Net  <== YOUR WORK
  #3  0.9214  HTU-Net (2024) — Hybrid Transformer U-Net
  #4  0.8700  YOLOv5 + Depthwise — SegNet (2025)
  #5  0.8392  Li et al. (2020) — Attention + MSP-cGAN
  >> your rank: #2 of 5
  saved /root/autodl-tmp/CBIS/figures/rank_inbreast_dice.png

  saved /root/autodl-tmp/CBIS/fig

## J · Data provenance and diagnostics


**original cell 17** — Show the crop cache folders and how they link to the CSVs ──  
<sub>1 output block(s) preserved</sub>


In [1]:
# ── Show the crop cache folders and how they link to the CSVs ──
import os, hashlib, pandas as pd
DATA_ROOT="/root/autodl-tmp/CBIS"
for folder in ["crop_cache_attn","crop_cache_tight","crop_cache_pad060"]:
    p=os.path.join(DATA_ROOT,folder)
    if os.path.exists(p):
        files=os.listdir(p)
        print(folder+": "+str(len(files))+" files | example: "+(files[0] if files else "empty"))
    else:
        print(folder+": (does not exist)")

print("\n── Verify the CSV->cache link for one lesion ──")
d=pd.read_csv(os.path.join(DATA_ROOT,"train_grouped.csv"))
row=d.iloc[0]
h=hashlib.md5(str(row["cropped_jpeg_path"]).encode()).hexdigest()
crop=os.path.join(DATA_ROOT,"crop_cache_attn",h+".png")
print("CSV row original path: "+str(row["cropped_jpeg_path"])[:70]+"...")
print("Maps to cached crop  : crop_cache_attn/"+h+".png")
print("That crop exists     : "+str(os.path.exists(crop)))

crop_cache_attn: 4442 files | example: 6ddb05c9391947d0f459cdbe472865f7.png
crop_cache_tight: 8884 files | example: 6ddb05c9391947d0f459cdbe472865f7_img.png
crop_cache_pad060: 3242 files | example: 6ddb05c9391947d0f459cdbe472865f7.png

── Verify the CSV->cache link for one lesion ──
CSV row original path: /root/autodl-tmp/CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.312534744711605661...
Maps to cached crop  : crop_cache_attn/6ddb05c9391947d0f459cdbe472865f7.png
That crop exists     : True


**original cell 86** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# Which crop folders exist, and which does the segmentation model fit?
# ══════════════════════════════════════════════════════════════════════
import os, glob, torch, numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda"); SZ=256
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

print("CROP FOLDERS ON DISK:")
for d in sorted(glob.glob(os.path.join(D,"crops*"))):
    n=len(glob.glob(os.path.join(d,"*_img.png")))
    print("  "+os.path.basename(d).ljust(28)+str(n)+" images")
print("\nCSV FILES:")
for f in sorted(glob.glob(os.path.join(D,"*.csv"))):
    print("  "+os.path.basename(f))

# test the seg model on each candidate crop folder
net=U(cat_skip_first=False, att_swap=False, duel=True).to(DEV)
sd=torch.load(os.path.join(D,"seg_cbis_mass.pth"),map_location="cpu")
if isinstance(sd,dict) and "state_dict" in sd: sd=sd["state_dict"]
net.load_state_dict(sd,strict=True); net.eval()

def test_folder(folder, n=60):
    imgs=sorted(glob.glob(os.path.join(folder,"*_img.png")))[:n]
    if not imgs: return None
    ds=[]
    for p in imgs:
        mp=p.replace("_img.png","_msk.png")
        if not os.path.exists(mp): continue
        im=cv2.imread(p,cv2.IMREAD_GRAYSCALE); mk=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
        if im is None or mk is None: continue
        x0=_clahe.apply(cv2.resize(im,(SZ,SZ))).astype(np.float32)/255.
        gt=(cv2.resize(mk,(SZ,SZ),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        with torch.no_grad():
            t=torch.from_numpy(x0[None,None]).float().to(DEV)
            pr=torch.softmax(net(t).float(),1)[0,1].cpu().numpy()
        m=(pr>0.5).astype(np.uint8); tot=m.sum()+gt.sum()
        ds.append(2*(m&gt).sum()/tot if tot>0 else 1.0)
    return (np.mean(ds), len(ds)) if ds else None

print("\n"+"="*58)
print("SEGMENTATION MODEL TESTED ON EACH CROP FOLDER")
print("="*58)
for d in sorted(glob.glob(os.path.join(D,"crops*"))):
    r=test_folder(d)
    if r: print("  "+os.path.basename(d).ljust(28)+"Dice "+format(r[0],".4f")+"   (n="+str(r[1])+")")
print("="*58)
print("the folder with the highest Dice is what the model was trained on")

CROP FOLDERS ON DISK:
  crops_clean512_calc         1866 images
  crops_clean512_calc_tightmask0 images
  crops_fixed_calc            1866 images
  crops_fixed_mass            1696 images
  crops_hires_calc            1866 images
  crops_tight512_calc         1866 images
  crops_v3                    3242 images
  crops_v4                    3242 images
  crops_v5                    3164 images
  crops_v6                    3566 images

CSV FILES:
  ATT_test_metrics.csv
  ATT_test_perimage.csv
  CV_calc_ALL_false_negatives.csv
  CV_calc_ALL_false_positives.csv
  FINAL_cv_calcification.csv
  FINAL_cv_mass.csv
  FINAL_operating_points.csv
  FINAL_test_metrics.csv
  FINAL_test_perimage.csv
  THESIS_final_metrics.csv
  THESIS_final_perimage.csv
  THESIS_perimage_with_metadata.csv
  ablation_plain_attnunet.csv
  ablation_plain_test_perimage.csv
  ablation_prep_aug.csv
  al_labeled_pool.csv
  al_unlabeled_pool.csv
  align_report.csv
  all_dataset_seg_summary.csv
  all_results_per_image.csv
 